# XAU Price Forecasting - ARIMA (BTC process)

Model workflow reference: `BTC_TEST/ARIMA_BTC.ipynb`. Use its Step 1-9 sequence, log Price, chronological Train 80% / Test 20%, Train-verified d=1, p,q=0..5 with no-drift/drift, AIC Top 3, supplementary Train-only auto_arima cross-check, Ljung-Box/root gates, robust reporting, fixed-parameter rolling Test and future forecasts, RMSE/MAPE and plots.

**XAU data contract:** `CSV/XAU_USD Historical Data.csv`; target `Price`; units **USD (source price)**. The source does not specify venue, fixing, timezone, adjustments or physical-unit metadata. Preserve observed dates without filling gaps. Horizons [1,3,5,7,14,21,30] count source observations. Future dates use the declared Mon-Fri pattern, not verified holiday sessions. Auxiliary OHLC/volume do not enter the univariate Price model.

**BTC selection rule:** choose the lowest-AIC Top 3 fit passing Ljung-Box at lags 10/20/30 and roots. No WTI-style residual-warning fallback is enabled. If none passes, report NO ACCEPTED MODEL and skip forecast/metric calculations without a traceback.

Run all cells in the Python 3.12 vs_nine_tf environment. Outputs: `OUTPUT/ARIMA_XAU/{tables,figures,summaries,models}`. Development uses Train only, with no validation and no all-data refit. This user-requested revision follows earlier Test inspection; the existing Test period is not a new untouched holdout.


### RMSE/MAPE — Results navigation

- [Task 1: Future forecast RMSE/MAPE](#task1-future-metrics) — section **7.5**, using supplied actual prices after the original dataset.
- [Task 2: Test set RMSE/MAPE](#task2-test-metrics) — under **Step 8 - RMSE and MAPE on Test**, using the original Test 20%.

Task 1 and Task 2 have separate CSV exports in `OUTPUT/METRICS_REVIEW/Task1_Future/ARIMA_XAU` and `Task2_Test/ARIMA_XAU`. Tables are already saved in this notebook. New reporting cells were executed against the saved forecasts; model-training cells and their saved outputs were preserved.


## Configuration & Library Versions


In [1]:
from pathlib import Path
import sys
import json
import warnings
import hashlib
from importlib.metadata import version

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import statsmodels
import pmdarima as pm
import scipy
from scipy import stats
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.stattools import jarque_bera
from IPython.display import display, Markdown, HTML

HORIZONS = [1, 3, 5, 7, 14, 21, 30]
DAILY_FORECAST_WINDOWS = [7, 30]
TRAIN_RATIO = 0.8
ALPHA = 0.05
ASSET = 'XAU'
UNIT = 'USD (source price)'
DATE_FORMAT = '%m/%d/%Y'
MAX_P = MAX_Q = 5
TOP_K = 3
LB_LAGS = [10, 20, 30]
WEEKMASK = 'Mon Tue Wed Thu Fri'
CALENDAR_NOTE = 'Horizons count successive observed records. Future dates use the source weekday pattern, not confirmed exchange sessions; source holidays/omissions and instrument details are unknown.'
REFERENCE_PATH = Path('BTC_TEST/ARIMA_BTC.ipynb')
search_dirs = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((base for base in search_dirs if (base/'CSV/XAU_USD Historical Data.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Launch Jupyter inside PROJECT_2026 or XAU_TEST.')
DATA_PATH = PROJECT_ROOT / 'CSV/XAU_USD Historical Data.csv'
REFERENCE_PATH = PROJECT_ROOT / REFERENCE_PATH
OUTPUT_DIR = PROJECT_ROOT / 'OUTPUT/ARIMA_XAU'
TABLE_DIR, FIGURE_DIR, SUMMARY_DIR, MODEL_DIR = [OUTPUT_DIR/k for k in ['tables','figures','summaries','models']]
for directory in (TABLE_DIR, FIGURE_DIR, SUMMARY_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)
assert HORIZONS == sorted(set(HORIZONS)) and max(DAILY_FORECAST_WINDOWS) <= max(HORIZONS)

plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True,
                     'grid.alpha': 0.25, 'figure.dpi': 110})
pd.set_option('display.max_columns', 20)

versions = pd.DataFrame([(m, version(m)) for m in
    ['pandas', 'numpy', 'matplotlib', 'statsmodels', 'pmdarima', 'scipy']],
    columns=['library', 'version'])
print('Python:', sys.version)
display(versions)
print('Data:', DATA_PATH, '\nOutputs:', TABLE_DIR.parent)


notebook_tables, notebook_figures = {}, []

def save_table(frame, filename):
    notebook_tables[filename] = frame.copy(deep=True)
    frame.to_csv(TABLE_DIR / filename, index=False, encoding='utf-8-sig')
    aliases = {'grid_search_aic_bic.csv':'arima_grid.csv', 'top3_candidates.csv':'arima_top_candidates.csv',
               'parameter_significance.csv':'arima_parameters.csv', 'model_roots.csv':'arima_roots.csv',
               'assumption_checks.csv':'train_residual_diagnostics.csv'}
    if filename in aliases:
        notebook_tables[aliases[filename]] = frame.copy(deep=True)
        frame.to_csv(TABLE_DIR / aliases[filename], index=False, encoding='utf-8-sig')

def save_figure(fig, filename):
    notebook_figures.append(filename)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)

def warning_text(items):
    return ' | '.join(dict.fromkeys(str(w.message) for w in items))

def write_json(path, value):
    path.write_text(json.dumps(value, indent=2, default=str, ensure_ascii=False), encoding='utf-8')

def model_history(series):
    # Preserve observation order without assigning Bitcoin's daily calendar to XAU.
    return pd.Series(series.to_numpy(dtype=float), index=pd.RangeIndex(len(series)), name=series.name)

def show_table(frame):
    display(HTML('<div style="max-height:420px;overflow:auto;border:1px solid #ddd;padding:8px">'
        + frame.to_html(index=False, float_format=lambda x: f'{x:.6f}') + '</div>'))

save_table(versions, 'library_versions.csv')
configuration = dict(asset=ASSET, data=str(DATA_PATH), target='Price', unit=UNIT, date_format=DATE_FORMAT,
    train_ratio=TRAIN_RATIO, horizons=HORIZONS, horizon_unit='observations', daily_forecast_windows=DAILY_FORECAST_WINDOWS,
    weekmask=WEEKMASK, calendar=CALENDAR_NOTE, transformation='log price', d=1,
    max_p=MAX_P, max_q=MAX_Q, top_k=TOP_K, alpha=ALPHA, ljung_box_lags=LB_LAGS,
    selection='BTC strict Top 3 by AIC; lowest-AIC candidate passing all Ljung-Box and roots gates; no residual-warning fallback',
    allow_residual_warning_fallback=False,
    reference=str(REFERENCE_PATH), reference_sha256=hashlib.sha256(REFERENCE_PATH.read_bytes()).hexdigest(),
    validation=False, refit_during_test=False, python_executable=sys.executable, python_version=sys.version)
write_json(SUMMARY_DIR/'configuration.json', configuration)
write_json(SUMMARY_DIR/'execution_audit.json', dict(completed=False, status='RUNNING', asset=ASSET,
    forecasts_generated=False, note='Old model/forecast files are not certified for this run.'))
(SUMMARY_DIR/'execution_summary.md').write_text('# XAU execution in progress\n\nPrior exports are not current results.\n', encoding='utf-8')
res = None
final_label = None
accepted_model = False
display(pd.DataFrame([(key,str(value)) for key,value in configuration.items()], columns=['setting','value']))

selection_mode = 'NO_ACCEPTED_MODEL'
train_diagnostics_passed = False
diagnostic_status = 'NO_ACCEPTED_MODEL'


Python: 3.12.14 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:37:13) [MSC v.1942 64 bit (AMD64)]


,library,version
0,pandas,2.3.3
1,numpy,2.5.3
2,matplotlib,3.11.1
3,statsmodels,0.15.0
4,pmdarima,2.1.1
5,scipy,1.18.1


Data: C:\Users\User\Desktop\PROJECT_2026\CSV\XAU_USD Historical Data.csv 
Outputs: C:\Users\User\Desktop\PROJECT_2026\OUTPUT\ARIMA_XAU


,setting,value
0,asset,XAU
1,data,C:\Users\User\Desktop\PROJECT_2026\CSV\XAU_USD...
2,target,Price
3,unit,USD (source price)
4,date_format,%m/%d/%Y
5,train_ratio,0.8
6,horizons,"[1, 3, 5, 7, 14, 21, 30]"
7,horizon_unit,observations
8,daily_forecast_windows,"[7, 30]"
9,weekmask,Mon Tue Wed Thu Fri


## Step 1 — Data Preparation

Apply BTC preparation checks to XAU source dates/prices, optional volume and OHLC. Validate Price, disclose source inconsistencies and date gaps, and retain observed prices unchanged.

In [2]:
raw = pd.read_csv(DATA_PATH, dtype=str, encoding='utf-8-sig')
assert {'Date','Price'}.issubset(raw.columns), 'Missing Date/Price columns'
df = raw.copy()
df['Date'] = pd.to_datetime(df['Date'], format=DATE_FORMAT, errors='raise')
assert df['Date'].notna().all() and not df['Date'].duplicated().any(), 'Invalid or duplicate source dates'
df = df.sort_values('Date').set_index('Date')
for column in ['Price','Open','High','Low']:
    if column in df:
        df[column] = pd.to_numeric(df[column].str.replace(',', '', regex=False), errors='raise')
        assert np.isfinite(df[column]).all(), f'Invalid numeric {column}'
assert df['Price'].gt(0).all() and df.index.is_unique and df.index.is_monotonic_increasing

# Auxiliary volume is missing in this source and is not used to model Price.
volume_text = df.get('Vol.', pd.Series('', index=df.index)).fillna('').str.strip().str.upper().str.replace(',', '', regex=False)
volume_present = ~volume_text.isin(['','-'])
parts = volume_text[volume_present].str.extract(r'^(?P<number>\d+(?:\.\d+)?)(?P<suffix>[KMB]?)$')
assert parts.notna().all().all(), 'Unrecognized nonmissing volume format'
df['Volume'] = np.nan
df.loc[volume_present, 'Volume'] = pd.to_numeric(parts['number']) * parts['suffix'].map({'':1.,'K':1e3,'M':1e6,'B':1e9})
b_rows = df.loc[volume_text.str.endswith('B')].copy()
save_table(b_rows.reset_index(), 'volume_B_rows.csv')
if 'Change %' in df:
    df['Change %'] = pd.to_numeric(df['Change %'].str.replace('%','',regex=False), errors='raise')/100
    change_difference = (df['Price'].pct_change() - df['Change %']).abs()
else:
    change_difference = pd.Series(np.nan, index=df.index)
ohlc_flag = pd.Series(False, index=df.index)
if {'Open','High','Low'}.issubset(df.columns):
    ohlc_flag = df['High'].lt(df[['Open','Low','Price']].max(axis=1)) | df['Low'].gt(df[['Open','High','Price']].min(axis=1))
save_table(df.loc[ohlc_flag].reset_index(), 'source_ohlc_flags.csv')
day_mismatch = (df['Day_of_Week'] != df.index.day_name()).sum() if 'Day_of_Week' in df else 0
calendar_offset = pd.offsets.CustomBusinessDay(weekmask=WEEKMASK)
expected_dates = pd.date_range(df.index[0], df.index[-1], freq=calendar_offset)
missing_dates = expected_dates.difference(df.index)
unexpected_dates = df.index.difference(expected_dates)
save_table(pd.DataFrame({'date':missing_dates}), 'calendar_missing_dates.csv')
save_table(pd.DataFrame({'date':unexpected_dates}), 'calendar_unexpected_dates.csv')
save_table(df.reset_index(), 'prepared_target.csv')
quality = pd.DataFrame([dict(rows=len(df), duplicate_dates=0, invalid_prices=0,
    missing_volume=int(df.Volume.isna().sum()), ohlc_flags=int(ohlc_flag.sum()), day_label_mismatches=int(day_mismatch),
    change_percent_mismatches=int(change_difference.gt(.001).sum()),
    missing_relative_to_weekmask=len(missing_dates), unexpected_weekdays=len(unexpected_dates),
    first_date=df.index[0], last_date=df.index[-1], source_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())])
save_table(quality, 'data_quality.csv')
log_price = np.log(df['Price']).rename('log_price')
print(f'Validated {len(df):,} observed XAU prices; {df.index[0].date()} to {df.index[-1].date()}. No source rows were filled or removed.')
print(f'Volume unavailable in {df.Volume.isna().sum()} rows; {ohlc_flag.sum()} OHLC row(s) flagged for review, retained, and not used as model inputs.')
print(CALENDAR_NOTE)
display(quality.T)
display(df.head())
if ohlc_flag.any():
    display(Markdown('**Auxiliary OHLC inconsistency retained for source review (Price is modeled univariately):**'))
    display(df.loc[ohlc_flag])


Validated 1,350 observed XAU prices; 2021-07-01 to 2026-09-09. No source rows were filled or removed.


Volume unavailable in 1350 rows; 11 OHLC row(s) flagged for review, retained, and not used as model inputs.
Horizons count successive observed records. Future dates use the source weekday pattern, not confirmed exchange sessions; source holidays/omissions and instrument details are unknown.


,0
rows,1350
duplicate_dates,0
invalid_prices,0
missing_volume,1350
ohlc_flags,11
day_label_mismatches,0
change_percent_mismatches,0
missing_relative_to_weekmask,5
unexpected_weekdays,0
first_date,2021-07-01 00:00:00


,Price,Open,High,Low,Vol.,Change %,Volume
Date,,,,,,,
2021-07-01,1776.60,1770.06,1783.07,1765.47,NaN,0.0038,NaN
2021-07-02,1786.79,1776.83,1795.40,1773.73,NaN,0.0057,NaN
2021-07-05,1791.61,1787.69,1795.69,1784.09,NaN,0.0027,NaN
2021-07-06,1796.69,1791.88,1815.34,1789.91,NaN,0.0028,NaN
2021-07-07,1803.41,1796.96,1810.13,1793.64,NaN,0.0037,NaN


**Auxiliary OHLC inconsistency retained for source review (Price is modeled univariately):**

,Price,Open,High,Low,Vol.,Change %,Volume
Date,,,,,,,
2022-06-13,1818.77,1875.80,1877.38,1818.90,NaN,-0.0279,NaN
2022-08-02,1759.75,1772.23,1788.24,1759.86,NaN,-0.0068,NaN
2022-12-26,1797.61,1798.57,1798.78,1798.04,NaN,-0.0002,NaN
2023-01-02,1823.69,1823.85,1823.94,1823.85,NaN,-0.0004,NaN
2023-06-02,1947.63,1978.14,1983.52,1947.67,NaN,-0.0153,NaN
2023-06-07,1939.63,1963.59,1970.26,1939.80,NaN,-0.0118,NaN
2023-12-20,2029.19,2040.34,2043.94,2029.37,NaN,-0.0054,NaN
2023-12-28,2064.86,2077.34,2088.54,2065.89,NaN,-0.0059,NaN
2024-07-31,2448.10,2410.09,2451.00,2410.17,NaN,0.0165,NaN


## Step 2 — EDA (Descriptive Only)


In [3]:
# First figure: raw XAU price over the full sample.
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df.index, df['Price'], color='tab:blue')
ax.set(title='XAU closing price ? full sample', ylabel='Price (source USD)', xlabel='Date')
save_figure(fig, '01_raw_price_volume.png')

fig, ax = plt.subplots()
ax.plot(log_price)
ax.set(title='Log XAU price', ylabel='Natural log of USD', xlabel='Date')
save_figure(fig, '02_log_price.png')

log_return = log_price.diff().dropna()
fig, ax = plt.subplots()
ax.plot(log_return, linewidth=0.7)
ax.axhline(0, color='black', linewidth=0.6)
ax.set(title='Observed-step log return', ylabel='Log return', xlabel='Date')
save_figure(fig, '03_log_return.png')

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(df['Price'], alpha=0.45, label='Price')
axes[0].plot(df['Price'].rolling(30).mean(), label='30-observation mean')
axes[0].set(ylabel='USD', title='30-observation rolling statistics'); axes[0].legend()
axes[1].plot(df['Price'].rolling(30).std(), color='tab:orange')
axes[1].set(ylabel='30-observation std (USD)', xlabel='Date')
save_figure(fig, '04_rolling_stats.png')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(log_return, bins=60, density=True, alpha=0.75)
axes[0].set(title='Log-return distribution', xlabel='Log return', ylabel='Density')
stats.probplot(log_return, dist='norm', plot=axes[1])
axes[1].set_title('Log-return normal QQ plot')
save_figure(fig, '05_return_distribution.png')


C:\Users\User\AppData\Local\Temp\ipykernel_22704\1610437506.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Descriptive Statistics

Price and observed-step log return, reported separately for the full sample,
the first 80% (Train), and the final 20% (Test), using the notebook's raw
chronological split before scaling or windowing. Columns identify each sample;
rows give units, dates, Count, Mean, SD, Min, Q1, Median, Q3, Max, Skewness,
and Excess kurtosis.

SD uses `ddof=1`; skewness and excess kurtosis use pandas' bias-corrected
estimators (normal-distribution excess kurtosis = 0). Log returns are natural-log
changes, not percentages, across successive observed dates. The first full-sample
return is undefined and excluded. Returns belong to their ending date, so the
first Test return uses the last Train price. Calendar gaps are not filled.
Full-sample and Test statistics are descriptive reporting only, not inputs to
model selection or fitting. The tables below were calculated from the supplied
source data without rerunning model training.


In [4]:
# Descriptive reporting only; this cell does not fit or configure a model.
def _make_descriptive_statistics(price, train_ratio):
    assert price.index.is_monotonic_increasing and price.index.is_unique
    assert price.notna().all() and np.isfinite(price).all() and price.gt(0).all()
    split = int(train_ratio * len(price))
    assert 0 < split < len(price)
    # Assign returns to their ending date; retain the Train-to-Test boundary return.
    log_return = np.log(price).diff()
    rows = []
    for variable, series, unit in [('Price', price, 'USD (source price)'),
                                    ('Log return', log_return, 'natural log; not percent')]:
        for sample, start, end in [('Full sample', 0, len(price)),
                                    ('Train', 0, split), ('Test', split, len(price))]:
            values = series.iloc[start:end].dropna()
            summary = values.describe(percentiles=[0.25, 0.5, 0.75])
            rows.append({'Variable': variable, 'Sample': sample, 'Unit': unit,
                'Start': str(values.index[0].date()), 'End': str(values.index[-1].date()),
                'Count': int(summary['count']), 'Mean': summary['mean'],
                'SD': summary['std'], 'Min': summary['min'], 'Q1': summary['25%'],
                'Median': summary['50%'], 'Q3': summary['75%'], 'Max': summary['max'],
                'Skewness': values.skew(), 'Excess kurtosis': values.kurt()})
    return pd.DataFrame(rows)

descriptive_statistics = _make_descriptive_statistics(df['Price'], TRAIN_RATIO)
del _make_descriptive_statistics
for _descriptive_variable in ['Price', 'Log return']:
    print(_descriptive_variable)
    display(descriptive_statistics.loc[
        descriptive_statistics['Variable'].eq(_descriptive_variable)
    ].drop(columns='Variable').set_index('Sample').T)
del _descriptive_variable


Price


Sample,Full sample,Train,Test
Unit,USD (source price),USD (source price),USD (source price)
Start,2021-07-01,2021-07-01,2025-08-28
End,2026-09-09,2025-08-27,2026-09-09
Count,1350,1080,270
Mean,2625.357296,2185.753407,4383.772852
SD,1002.482149,496.511872,414.187003
Min,1621.57,1621.57,3417.2
Q1,1850.2075,1817.385,4078.3575
Median,2040.835,1962.485,4353.495
Q3,3317.14,2456.67,4664.35


Log return


Sample,Full sample,Train,Test
Unit,natural log; not percent,natural log; not percent,natural log; not percent
Start,2021-07-02,2021-07-02,2025-08-28
End,2026-09-09,2025-08-27,2026-09-09
Count,1349,1079,270
Mean,0.000673,0.000601,0.000959
SD,0.011401,0.009249,0.017564
Min,-0.103497,-0.03552,-0.103497
Q1,-0.004859,-0.004543,-0.007235
Median,0.000786,0.000586,0.002639
Q3,0.006856,0.005719,0.010419


## Step 3 — Chronological Train/Test Split


In [5]:
n = len(df)
n_train = int(TRAIN_RATIO * n)
n_test = n - n_train
train_price = df['Price'].iloc[:n_train].copy()
test_price = df['Price'].iloc[n_train:].copy()
train_log = log_price.iloc[:n_train].copy()
assert n_train+n_test == n and n_test >= max(HORIZONS)
assert train_log.index[-1] < test_price.index[0]
train_model_log = model_history(train_log)
split_summary = pd.DataFrame([dict(split=name, start_date=x.index[0].date(),
    end_date=x.index[-1].date(), n_observations=len(x), min=x.min(), max=x.max(), mean=x.mean())
    for name, x in [('Train', train_price), ('Test', test_price)]])
save_table(split_summary, 'split_summary.csv')
display(split_summary)
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(train_price, label='Train (80%)', color='tab:blue')
ax.plot(test_price, label='Test (20%)', color='tab:orange')
ax.axvspan(df.index[0], test_price.index[0], color='tab:blue', alpha=0.07)
ax.axvspan(test_price.index[0], df.index[-1], color='tab:orange', alpha=0.10)
ax.axvline(test_price.index[0], color='black', linestyle='--', label=f'Test starts {test_price.index[0].date()}')
ax.set(title='Chronological split — raw price', ylabel='Price (USD)', xlabel='Date')
ax.legend()
save_figure(fig, '06_train_test_split.png')


,split,start_date,end_date,n_observations,min,max,mean
0,Train,2021-07-01,2025-08-27,1080,1621.57,3432.64,2185.753407
1,Test,2025-08-28,2026-09-09,270,3417.20,5400.25,4383.772852


C:\Users\User\AppData\Local\Temp\ipykernel_22704\1610437506.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 4 — Stationarity (Train Only)

As in BTC, verify nonstationary log levels and stationary first differences jointly with ADF/KPSS before using d=1. Do not substitute another d based on Test.

In [6]:
train_diff = train_log.diff().dropna()
stationarity_rows = []
for series_name, values in [('log_price', train_log), ('diff_log_price', train_diff)]:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        adf = adfuller(values, autolag='AIC')
    stationarity_rows.append(dict(series=series_name, test='ADF', statistic=adf[0],
        p_value=adf[1], critical_values=json.dumps(adf[4]),
        conclusion='stationary' if adf[1] < ALPHA else 'non-stationary (fail to reject H0)',
        stationary=adf[1] < ALPHA, warnings=warning_text(caught)))
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        kp = kpss(values, regression='c', nlags='auto')
    stationarity_rows.append(dict(series=series_name, test='KPSS', statistic=kp[0],
        p_value=kp[1], critical_values=json.dumps(kp[3]),
        conclusion='stationary (fail to reject H0)' if kp[1] > ALPHA else 'non-stationary',
        stationary=kp[1] > ALPHA, warnings=warning_text(caught)))
stationarity_tests = pd.DataFrame(stationarity_rows)
save_table(stationarity_tests, 'stationarity_tests.csv')
display(stationarity_tests)
level_stationary = stationarity_tests.loc[stationarity_tests['series'].eq('log_price'), 'stationary'].all()
diff_stationary = stationarity_tests.loc[stationarity_tests['series'].eq('diff_log_price'), 'stationary'].all()
assert not level_stationary and diff_stationary, 'Train ADF/KPSS do not jointly support the specified d=1.'
d = 1
print('Train stationarity evidence supports d =', d)


,series,test,statistic,p_value,critical_values,conclusion,stationary,warnings
0,log_price,ADF,0.898343,0.993069,"{""1%"": -3.436436308605049, ""5%"": -2.8642273405...",non-stationary (fail to reject H0),False,adfuller currently returns a plain tuple whose...
1,log_price,KPSS,4.575717,0.010000,"{""10%"": 0.347, ""5%"": 0.463, ""2.5%"": 0.574, ""1%...",non-stationary,False,The test statistic is outside of the range of ...
2,diff_log_price,ADF,-24.854400,0.000000,"{""1%"": -3.436436308605049, ""5%"": -2.8642273405...",stationary,True,adfuller currently returns a plain tuple whose...
3,diff_log_price,KPSS,0.308871,0.100000,"{""10%"": 0.347, ""5%"": 0.463, ""2.5%"": 0.574, ""1%...",stationary (fail to reject H0),True,The test statistic is outside of the range of ...


Train stationarity evidence supports d = 1


## Step 5 — Model Identification (Train Only)

BTC grid: p,q=0..5, d=1, no-drift/drift. Rank converged finite-AIC fits by AIC/BIC/model and retain Top 3. auto_arima is a supplementary Train-only cross-check, not a replacement selection rule.

**AIC Rank:** อันดับเริ่มจาก 1 เฉพาะโมเดลที่ converged, AIC finite และไม่มี fit error เรียง AIC จากน้อยไปมาก แล้วใช้ BIC และชื่อโมเดลเมื่อค่าเท่ากัน ตารางด้านล่างแสดง **Top 5** เพื่อเปรียบเทียบ โดยยังใช้ **Top 3** สำหรับการเลือกโมเดลใน Step 6 ตามกติกาเดิมของ notebook นี้ ไม่เปลี่ยนผลพยากรณ์


In [7]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(train_diff, lags=40, ax=axes[0])
plot_pacf(train_diff, lags=40, method='ywm', ax=axes[1])
axes[0].set_title('ACF — differenced log price (train only)')
axes[1].set_title('PACF — differenced log price (train only)')
save_figure(fig, '07_acf_pacf.png')

grid_rows, fitted_models = [], {}
for p in range(MAX_P + 1):
    for q in range(MAX_Q + 1):
        for trend in ['n', 't']:
            label = f'ARIMA({p},{d},{q}), trend={trend}'
            row = dict(model=label, p=p, d=d, q=q, trend=trend, AIC=np.nan,
                       BIC=np.nan, converged=False, warnings='', error='')
            caught = []
            try:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always')
                    result = ARIMA(train_model_log, order=(p, d, q), trend=trend).fit()
                row.update(AIC=result.aic, BIC=result.bic,
                           converged=bool(result.mle_retvals.get('converged', False)))
                fitted_models[label] = result
            except Exception as exc:
                row['error'] = f'{type(exc).__name__}: {exc}'
            row['warnings'] = warning_text(caught)
            grid_rows.append(row)
    print(f'Grid progress: p={p}, {len(grid_rows)}/72 fits completed', flush=True)
grid_search = pd.DataFrame(grid_rows).sort_values(['AIC', 'BIC', 'model'], na_position='last').reset_index(drop=True)
# Sequential AIC ranks among converged, finite-AIC, error-free Train fits.
eligible_mask = grid_search['converged'] & np.isfinite(grid_search['AIC']) & grid_search['error'].eq('')
grid_search.insert(0, 'AIC Rank', pd.Series(pd.NA, index=grid_search.index, dtype='Int64'))
grid_search.loc[eligible_mask, 'AIC Rank'] = np.arange(1, int(eligible_mask.sum()) + 1)
save_table(grid_search, 'grid_search_aic_bic.csv')
eligible = grid_search.loc[eligible_mask].copy()
assert len(eligible) >= 3, 'Fewer than three converged grid candidates; inspect grid_search_aic_bic.csv'
candidates = eligible.head(3).copy()
save_table(candidates, 'top3_candidates.csv')
# Show Top 5 for comparison; the original Top 3 selection is unchanged.
aic_rank_preview = eligible.head(5).copy()
display(aic_rank_preview.style.format({'AIC': '{:.6f}', 'BIC': '{:.6f}'}).hide(axis='index'))


C:\Users\User\AppData\Local\Temp\ipykernel_22704\1610437506.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Grid progress: p=0, 12/72 fits completed


Grid progress: p=1, 24/72 fits completed


Grid progress: p=2, 36/72 fits completed


Grid progress: p=3, 48/72 fits completed


Grid progress: p=4, 60/72 fits completed


Grid progress: p=5, 72/72 fits completed


AIC Rank,model,p,d,q,trend,AIC,BIC,converged,warnings,error
1,"ARIMA(2,1,0), trend=t",2,1,0,t,-7042.017943,-7022.082783,True,,
2,"ARIMA(0,1,2), trend=t",0,1,2,t,-7041.776032,-7021.840872,True,,
3,"ARIMA(0,1,0), trend=t",0,1,0,t,-7041.466921,-7031.499341,True,,
4,"ARIMA(3,1,0), trend=t",3,1,0,t,-7040.256870,-7015.337920,True,,
5,"ARIMA(0,1,3), trend=t",0,1,3,t,-7040.045280,-7015.126330,True,,


In [8]:
cross_rows = []
for intercept in [False, True]:
    caught = []
    row = dict(with_intercept=intercept, order='', AIC=np.nan, BIC=np.nan,
               converged=False, warnings='', error='')
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            cross = pm.auto_arima(train_model_log, d=1, seasonal=False, stepwise=False,
                information_criterion='aic', start_p=0, start_q=0,
                max_p=MAX_P, max_q=MAX_Q, max_order=None, with_intercept=intercept,
                suppress_warnings=False, error_action='warn', n_jobs=1)
        row.update(order=str(cross.order), AIC=cross.aic(), BIC=cross.bic(),
                   converged=bool(cross.arima_res_.mle_retvals.get('converged', False)))
    except Exception as exc:
        row['error'] = f'{type(exc).__name__}: {exc}'
    row['warnings'] = warning_text(caught)
    cross_rows.append(row)
    print(f'auto_arima cross-check complete: with_intercept={intercept}', flush=True)
cross_check = pd.DataFrame(cross_rows)
save_table(cross_check, 'auto_arima_cross_check.csv')
display(cross_check)
assert cross_check['error'].eq('').all(), 'auto_arima cross-check failed; inspect saved errors'


auto_arima cross-check complete: with_intercept=False


auto_arima cross-check complete: with_intercept=True


,with_intercept,order,AIC,BIC,converged,warnings,error
0,False,"(0, 1, 0)",-7038.918342,-7033.934552,True,Maximum Likelihood optimization failed to conv...,
1,True,"(2, 1, 0)",-7042.017949,-7022.082789,True,Maximum Likelihood optimization failed to conv...,


## Step 6 — Residual Diagnostics & Final Model (Train Only)

Use BTC's strict Top 3 AIC shortlist. Remove initialization burn-in, then require Ljung-Box p > 0.05 at all lags 10/20/30 (model_df=p+q) and existing AR/MA roots outside the unit circle. Choose the lowest-AIC candidate passing both. If none passes, report NO ACCEPTED MODEL and skip forecast/metric calculations. No residual-warning fallback is enabled for XAU.

Mean, ARCH, normality and parameter significance remain reporting-only. Significant ARCH triggers robust covariance reporting; forecasts retain the original fitted parameters. Passing selection gates does not establish forecast accuracy or interval calibration.


In [9]:
check_rows, parameter_rows, root_rows, candidate_status = [], [], [], []
robust_results, robust_rows, robust_warning_messages = {}, [], {}
def add_check(model, test, statistic=np.nan, p_value=np.nan, passed=None, note=''):
    check_rows.append(dict(model=model, test=test, statistic=statistic, p_value=p_value,
                           pass_fail='REPORTED' if passed is None else ('PASS' if passed else 'FAIL'), note=note))

for item in candidates.itertuples(index=False):
    result = fitted_models[item.model]
    burn_in = max(d, int(result.loglikelihood_burn), int(getattr(result.filter_results,'nobs_diffuse',0)))
    residuals = pd.Series(np.asarray(result.resid)[burn_in:]).dropna()
    lb = acorr_ljungbox(residuals, lags=[10, 20, 30], model_df=item.p + item.q, return_df=True)
    lb_pass = bool(lb['lb_pvalue'].notna().all() and (lb['lb_pvalue'] > ALPHA).all())
    for lag, row in lb.iterrows():
        add_check(item.model, f'Ljung-Box lag {lag}', row['lb_stat'], row['lb_pvalue'],
                  bool(row['lb_pvalue'] > ALPHA), f'model_df={item.p + item.q}')
    mean_stat, mean_p = stats.ttest_1samp(residuals, 0)
    add_check(item.model, 'Mean = 0 (t-test)', mean_stat, mean_p, mean_p > ALPHA,
              f'Residual mean={residuals.mean():.6g}')
    arch_stat, arch_p, _, _ = het_arch(residuals, nlags=10)
    add_check(item.model, 'ARCH LM (10 lags)', arch_stat, arch_p, arch_p > ALPHA)
    if arch_p <= ALPHA:
        with warnings.catch_warnings(record=True) as robust_caught:
            warnings.simplefilter('always')
            robust = ARIMA(train_model_log, order=(item.p, d, item.q), trend=item.trend).fit(cov_type='robust')
        robust_results[item.model] = robust
        robust_warning_messages[item.model] = warning_text(robust_caught)
        for name, coef, se, pv in zip(robust.param_names, robust.params, robust.bse, robust.pvalues):
            robust_rows.append(dict(model=item.model, parameter=name, estimate=coef,
                robust_standard_error=se, robust_p_value=pv,
                converged=bool(robust.mle_retvals.get('converged', False)),
                warnings=warning_text(robust_caught)))
    jb_stat, jb_p, skew, kurt = jarque_bera(residuals)
    add_check(item.model, 'Jarque-Bera', jb_stat, jb_p, jb_p > ALPHA,
              f'skew={skew:.4f}, kurtosis={kurt:.4f}')
    roots_pass = True
    for kind, roots in [('AR', result.arroots), ('MA', result.maroots)]:
        moduli = np.abs(roots)
        passed = bool(np.isfinite(moduli).all() and (moduli > 1).all())
        roots_pass &= passed
        add_check(item.model, f'{kind} roots outside unit circle',
                  moduli.min() if len(moduli) else np.nan, passed=passed if len(moduli) else None,
                  note='N/A: no roots' if not len(moduli) else f'All |root|>1: {passed}')
        for j, root in enumerate(roots, 1):
            root_rows.append(dict(model=item.model, kind=kind, root_number=j,
                                  real=root.real, imag=root.imag, modulus=abs(root), outside=abs(root) > 1))
    for name, coef, se, pv in zip(result.param_names, result.params, result.bse, result.pvalues):
        parameter_rows.append(dict(model=item.model, parameter=name, estimate=coef, standard_error=se,
                                   p_value=pv, significant=pv < ALPHA, covariance=result.cov_type))
        add_check(item.model, f'Parameter significance: {name}', coef, pv, pv < ALPHA,
                  'Report only; estimate in statistic column, significance uses p < alpha')
    candidate_status.append(dict(model=item.model, AIC=item.AIC, BIC=item.BIC, ljung_box_pass=lb_pass,
        roots_pass=roots_pass, arch_pvalue=arch_p,
        **{f'lb_pvalue_{lag}':float(lb.loc[lag,'lb_pvalue']) for lag in LB_LAGS}))

assumption_checks = pd.DataFrame(check_rows)
parameter_significance = pd.DataFrame(parameter_rows)
candidate_diagnostics = pd.DataFrame(candidate_status).sort_values('AIC')
save_table(assumption_checks, 'assumption_checks.csv')
save_table(parameter_significance, 'parameter_significance.csv')
save_table(pd.DataFrame(root_rows, columns=['model','kind','root_number','real','imag','modulus','outside']), 'model_roots.csv')
save_table(candidate_diagnostics, 'candidate_diagnostics.csv')
if robust_rows:
    save_table(pd.DataFrame(robust_rows), 'robust_parameter_significance.csv')
show_table(candidate_diagnostics)
show_table(assumption_checks)
passing = candidate_diagnostics.loc[candidate_diagnostics['ljung_box_pass'] & candidate_diagnostics['roots_pass']].copy()
accepted_model = not passing.empty
train_diagnostics_passed = accepted_model
selection_mode = 'STRICT_DIAGNOSTICS' if accepted_model else 'NO_ACCEPTED_MODEL'
diagnostic_status = 'TRAIN_LJUNG_BOX_AND_ROOTS_PASS' if accepted_model else 'NO_ACCEPTED_MODEL'
candidate_diagnostics['selected'] = False
if accepted_model:
    candidate_diagnostics.loc[candidate_diagnostics.model.eq(passing.iloc[0]['model']), 'selected'] = True
candidate_diagnostics['selection_mode'] = selection_mode
save_table(candidate_diagnostics, 'candidate_diagnostics.csv')
if accepted_model:
    final_label = passing.iloc[0]['model']
    final_spec = candidates.set_index('model').loc[final_label]
    res = fitted_models[final_label]  # reuse original train fit; frozen forecast parameters
    frozen_params = np.asarray(res.params).copy()
    assert res.nobs == n_train
    print('Selected:', final_label, '; selection mode:', selection_mode)
    display(parameter_significance.loc[parameter_significance['model'].eq(final_label)])
    fig = res.plot_diagnostics(figsize=(13, 9))
    fig.suptitle(f'{final_label} — train diagnostics', y=1.02)
    save_figure(fig, '08_residual_diagnostics.png')
    
    summary_text = f'Selection mode: {selection_mode}; diagnostic status: {diagnostic_status}\n\n' + res.summary().as_text()
    robust_res = None
    if passing.iloc[0]['arch_pvalue'] <= ALPHA:
        robust_res = robust_results[final_label]
        display(pd.DataFrame(robust_rows).loc[lambda x: x['model'].eq(final_label)])
        summary_text += '\n\nROBUST COVARIANCE REFIT ON TRAIN — reporting only; forecasts retain original parameters.\n'
        summary_text += robust_res.summary().as_text()
        summary_text += '\nWarnings: ' + robust_warning_messages[final_label]
        summary_text += '\nRobust refit converged: ' + str(robust_res.mle_retvals.get('converged', False))
    assert np.array_equal(np.asarray(res.params), frozen_params)
    (SUMMARY_DIR / 'final_model_summary.txt').write_text(summary_text, encoding='utf-8')
    print(summary_text)
    
    accepted_model = True
    res.save(MODEL_DIR/'arima_train.pkl')
    write_json(MODEL_DIR/'selected_order.json', dict(order=[int(final_spec.p),d,int(final_spec.q)], trend=final_spec.trend,
        parameters=frozen_params.tolist(), fit_end_date=train_log.index[-1], fit_n=n_train,
        selection=configuration['selection'], selection_mode=selection_mode,
        train_diagnostics_passed=train_diagnostics_passed, diagnostic_status=diagnostic_status))
else:

    stop_message = 'None of the BTC-style AIC Top 3 passes both Train Ljung-Box at lags 10/20/30 and roots. No final model is accepted.'
    display(Markdown('**Selection status: NO ACCEPTED MODEL**\n\n'+stop_message+
        '\n\nSteps 7-8 will show a skipped status; Step 9 will summarize the actual diagnostic evidence. No forecasts or metrics are fabricated.'))
    write_json(SUMMARY_DIR/'execution_audit.json', dict(completed=False, status='NO_ACCEPTED_MODEL',
        asset=ASSET, model_accepted=False, forecasts_generated=False, stopped_selection_step=6,
        reason=stop_message, candidate_diagnostics=candidate_diagnostics.to_dict('records'),
        note='Prior model/forecast exports are not current results.'))


C:\Users\User\anaconda3\envs\vs_nine_tf\Lib\site-packages\statsmodels\stats\diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(


C:\Users\User\anaconda3\envs\vs_nine_tf\Lib\site-packages\statsmodels\stats\diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(


C:\Users\User\anaconda3\envs\vs_nine_tf\Lib\site-packages\statsmodels\stats\diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(


model,AIC,BIC,ljung_box_pass,roots_pass,arch_pvalue,lb_pvalue_10,lb_pvalue_20,lb_pvalue_30
"ARIMA(2,1,0), trend=t",-7042.017943,-7022.082783,True,True,0.000000,0.928211,0.564085,0.375138
"ARIMA(0,1,2), trend=t",-7041.776032,-7021.840872,True,True,0.000000,0.912813,0.542422,0.362173
"ARIMA(0,1,0), trend=t",-7041.466921,-7031.499341,True,True,0.000000,0.678470,0.356224,0.318482


model,test,statistic,p_value,pass_fail,note
"ARIMA(2,1,0), trend=t",Ljung-Box lag 10,3.095667,0.928211,PASS,model_df=2
"ARIMA(2,1,0), trend=t",Ljung-Box lag 20,16.408158,0.564085,PASS,model_df=2
"ARIMA(2,1,0), trend=t",Ljung-Box lag 30,29.752650,0.375138,PASS,model_df=2
"ARIMA(2,1,0), trend=t",Mean = 0 (t-test),0.011710,0.990659,PASS,Residual mean=3.29012e-06
"ARIMA(2,1,0), trend=t",ARCH LM (10 lags),70.370106,0.000000,FAIL,
"ARIMA(2,1,0), trend=t",Jarque-Bera,78.259241,0.000000,FAIL,"skew=0.0182, kurtosis=4.3189"
"ARIMA(2,1,0), trend=t",AR roots outside unit circle,3.960369,NaN,PASS,All |root|>1: True
"ARIMA(2,1,0), trend=t",MA roots outside unit circle,NaN,NaN,REPORTED,N/A: no roots
"ARIMA(2,1,0), trend=t",Parameter significance: x1,0.000597,0.023511,PASS,"Report only; estimate in statistic column, significance uses p < alpha"
"ARIMA(2,1,0), trend=t",Parameter significance: ar.L1,-0.013294,0.617980,FAIL,"Report only; estimate in statistic column, significance uses p < alpha"


Selected: ARIMA(2,1,0), trend=t ; selection mode: STRICT_DIAGNOSTICS


,model,parameter,estimate,standard_error,p_value,significant,covariance
0,"ARIMA(2,1,0), trend=t",x1,0.000597,0.000264,2.351098e-02,True,opg
1,"ARIMA(2,1,0), trend=t",ar.L1,-0.013294,0.026656,6.179803e-01,False,opg
2,"ARIMA(2,1,0), trend=t",ar.L2,-0.063757,0.025673,1.301188e-02,True,opg
3,"ARIMA(2,1,0), trend=t",sigma2,0.000085,0.000003,1.755150e-196,True,opg


C:\Users\User\AppData\Local\Temp\ipykernel_22704\1610437506.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,model,parameter,estimate,robust_standard_error,robust_p_value,converged,warnings
0,"ARIMA(2,1,0), trend=t",x1,0.000597,0.000261,2.200034e-02,True,
1,"ARIMA(2,1,0), trend=t",ar.L1,-0.013294,0.034738,7.019535e-01,True,
2,"ARIMA(2,1,0), trend=t",ar.L2,-0.063757,0.036295,7.898028e-02,True,
3,"ARIMA(2,1,0), trend=t",sigma2,0.000085,0.000005,1.619792e-72,True,


Selection mode: STRICT_DIAGNOSTICS; diagnostic status: TRAIN_LJUNG_BOX_AND_ROOTS_PASS

                               SARIMAX Results                                
Dep. Variable:              log_price   No. Observations:                 1080
Model:                 ARIMA(2, 1, 0)   Log Likelihood                3525.009
Date:                Mon, 02 Nov 2026   AIC                          -7042.018
Time:                        22:05:18   BIC                          -7022.083
Sample:                             0   HQIC                         -7034.469
                               - 1080                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
x1             0.0006      0.000      2.265      0.024    8.05e-05       0.001
ar.L1         -0.0133      0.027     -0.499 

## Step 7 - Forecast Values by Horizon
The next cells produce forecasts using the parameters estimated on train.
`arima_fc` is the point forecast in USD; `arima_lo95` and `arima_hi95` are its model-based 95% prediction-interval bounds.
The exp-transformed log forecast mean is a conditional median on the USD scale.

### 7.1 Rolling-origin forecasts for the test period
Each origin uses observations up to that date only, with `refit=False`.

In [10]:
if not accepted_model:
    display(Markdown('**Skipped:** Step 6 accepted no model. No forecast or metric is generated by this cell.'))
else:
    log_all = log_price
    forecast_rows = []
    for i in range(n_train - 1, n - 1):
        r = res.apply(model_history(log_all.iloc[:i+1]), refit=False)
        assert np.array_equal(np.asarray(r.params), frozen_params), 'Parameters changed during test'
        fc = r.get_forecast(steps=max(HORIZONS))
        means = np.asarray(fc.predicted_mean)
        ci = np.asarray(fc.conf_int(alpha=ALPHA))
        for h in HORIZONS:
            target = i + h
            if target < n:
                row = dict(origin_date=log_all.index[i], h=h, target_date=log_all.index[target],
                    actual=df['Price'].iloc[target], arima_fc=np.exp(means[h-1]),
                    arima_lo95=np.exp(ci[h-1, 0]), arima_hi95=np.exp(ci[h-1, 1]),
                    price_origin=df['Price'].iloc[i])
                # Treat rounding noise as zero price movement, not a predicted direction.
                if np.isclose(row['arima_fc'], row['price_origin'], rtol=32*np.finfo(float).eps, atol=0):
                    row['arima_fc'] = row['price_origin']
                forecast_rows.append(row)
        if (i - n_train + 2) % 50 == 0:
            print(f'Rolling origins completed: {i - n_train + 2}/{n_test}', flush=True)
    forecasts = pd.DataFrame(forecast_rows).sort_values(['h', 'target_date']).reset_index(drop=True)
    counts = forecasts.groupby('h').size()
    for h in HORIZONS:
        assert counts[h] == n_test - h + 1, (h, counts[h])
    assert (forecasts['target_date'] >= test_price.index[0]).all()
    assert (df.index.get_indexer(forecasts.target_date)-df.index.get_indexer(forecasts.origin_date) == forecasts.h.to_numpy()).all()
    assert not forecasts.duplicated(['origin_date', 'h']).any()
    assert np.isfinite(forecasts.select_dtypes('number').to_numpy()).all()
    assert (forecasts['arima_lo95'] <= forecasts['arima_hi95']).all()
    forecasts['diagnostic_status'] = diagnostic_status
    save_table(forecasts, 'forecasts_long.csv')
    display(counts.rename('n_points').to_frame())
    
    assert (forecasts[['actual','arima_fc','arima_lo95','arima_hi95']] > 0).all().all()
    assert (forecasts.arima_lo95 <= forecasts.arima_fc).all() and (forecasts.arima_fc <= forecasts.arima_hi95).all()


Rolling origins completed: 50/270


Rolling origins completed: 100/270


Rolling origins completed: 150/270


Rolling origins completed: 200/270


Rolling origins completed: 250/270


,n_points
h,
1,270
3,268
5,266
7,264
14,257
21,250
30,241


### 7.2 Test forecast tables - h = 1, 3, 5, 7, 14, 21, 30
Each table displays every forecast at that horizon: origin date, target date, price at origin, actual target price, and forecast in USD.
Scroll inside a table to inspect all dates. Full tables are also exported to `OUTPUT/ARIMA_BTC/tables/forecasts_h{h}.csv`.

In [11]:
if not accepted_model:
    display(Markdown('**Skipped:** Step 6 accepted no model. No forecast or metric is generated by this cell.'))
else:
    # All test forecasts are visible in scrollable tables; export all rows per horizon.
    forecast_columns = ['origin_date', 'target_date', 'price_origin', 'actual',
                        'arima_fc', 'arima_lo95', 'arima_hi95', 'diagnostic_status']
    for h in HORIZONS:
        horizon_forecasts = forecasts.loc[forecasts['h'].eq(h), forecast_columns].copy()
        save_table(horizon_forecasts, f'forecasts_h{h}.csv')
        display(Markdown(f'**Test forecasts: h = {h} observations ({len(horizon_forecasts)} rows)**'))
        table_html = horizon_forecasts.to_html(index=False, float_format=lambda x: f'{x:,.2f}')
        display(HTML('<div style="max-height:360px;overflow:auto;border:1px solid #ddd;padding:8px">'
                     + table_html + '</div>'))


**Test forecasts: h = 1 observations (270 rows)**

origin_date,target_date,price_origin,actual,arima_fc,arima_lo95,arima_hi95,diagnostic_status
2025-08-27,2025-08-28,"3,397.42","3,417.20","3,397.87","3,337.03","3,459.83",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-08-28,2025-08-29,"3,417.20","3,448.00","3,418.84","3,357.62","3,481.18",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-08-29,2025-09-01,"3,448.00","3,476.47","3,448.53","3,386.78","3,511.41",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-01,2025-09-02,"3,476.47","3,533.73","3,476.34","3,414.08","3,539.73",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-02,2025-09-03,"3,533.73","3,559.43","3,533.38","3,470.11","3,597.81",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-03,2025-09-04,"3,559.43","3,546.17","3,557.67","3,493.96","3,622.54",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-04,2025-09-05,"3,546.17","3,586.81","3,546.99","3,483.47","3,611.67",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-05,2025-09-08,"3,586.81","3,635.84","3,589.43","3,525.15","3,654.88",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-08,2025-09-09,"3,635.84","3,626.48","3,634.88","3,569.79","3,701.16",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-09,2025-09-10,"3,626.48","3,640.74","3,625.80","3,560.87","3,691.91",TRAIN_LJUNG_BOX_AND_ROOTS_PASS


**Test forecasts: h = 3 observations (268 rows)**

origin_date,target_date,price_origin,actual,arima_fc,arima_lo95,arima_hi95,diagnostic_status
2025-08-27,2025-09-01,"3,397.42","3,476.47","3,401.90","3,300.12","3,506.82",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-08-28,2025-09-02,"3,417.20","3,533.73","3,421.84","3,319.46","3,527.37",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-08-29,2025-09-03,"3,448.00","3,559.43","3,450.95","3,347.70","3,557.38",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-01,2025-09-04,"3,476.47","3,546.17","3,478.99","3,374.91","3,586.29",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-02,2025-09-05,"3,533.73","3,586.81","3,534.30","3,428.55","3,643.30",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-03,2025-09-08,"3,559.43","3,635.84","3,560.73","3,454.20","3,670.55",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-04,2025-09-09,"3,546.17","3,626.48","3,552.30","3,446.02","3,661.85",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-05,2025-09-10,"3,586.81","3,640.74","3,591.24","3,483.80","3,702.00",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-08,2025-09-11,"3,635.84","3,634.09","3,636.50","3,527.70","3,748.65",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-09,2025-09-12,"3,626.48","3,643.13","3,631.08","3,522.44","3,743.06",TRAIN_LJUNG_BOX_AND_ROOTS_PASS


**Test forecasts: h = 5 observations (266 rows)**

origin_date,target_date,price_origin,actual,arima_fc,arima_lo95,arima_hi95,diagnostic_status
2025-08-27,2025-09-03,"3,397.42","3,559.43","3,405.97","3,277.27","3,539.72",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-08-28,2025-09-04,"3,417.20","3,546.17","3,426.00","3,296.54","3,560.54",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-08-29,2025-09-05,"3,448.00","3,586.81","3,455.18","3,324.63","3,590.87",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-01,2025-09-08,"3,476.47","3,635.84","3,483.25","3,351.63","3,620.03",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-02,2025-09-09,"3,533.73","3,626.48","3,538.73","3,405.01","3,677.69",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-03,2025-09-10,"3,559.43","3,640.74","3,565.06","3,430.35","3,705.06",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-04,2025-09-11,"3,546.17","3,634.09","3,556.47","3,422.09","3,696.14",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-05,2025-09-12,"3,586.81","3,643.13","3,595.69","3,459.83","3,736.89",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-08,2025-09-15,"3,635.84","3,679.13","3,641.01","3,503.43","3,783.99",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-09,2025-09-16,"3,626.48","3,689.80","3,635.36","3,497.99","3,778.11",TRAIN_LJUNG_BOX_AND_ROOTS_PASS


... (4 outputs left out)

**Test forecasts: h = 21 observations (250 rows)**

origin_date,target_date,price_origin,actual,arima_fc,arima_lo95,arima_hi95,diagnostic_status
2025-08-27,2025-09-25,"3,397.42","3,749.37","3,438.67","3,182.63","3,715.32",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-08-28,2025-09-26,"3,417.20","3,760.24","3,458.90","3,201.35","3,737.17",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-08-29,2025-09-29,"3,448.00","3,833.93","3,488.36","3,228.62","3,769.00",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-01,2025-09-30,"3,476.47","3,858.51","3,516.69","3,254.84","3,799.61",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-02,2025-10-01,"3,533.73","3,865.80","3,572.70","3,306.67","3,860.12",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-03,2025-10-02,"3,559.43","3,856.53","3,599.29","3,331.29","3,888.86",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-04,2025-10-03,"3,546.17","3,886.83","3,590.63","3,323.27","3,879.50",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-05,2025-10-06,"3,586.81","3,961.09","3,630.21","3,359.91","3,922.27",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-08,2025-10-07,"3,635.84","3,983.82","3,675.97","3,402.26","3,971.70",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-09,2025-10-08,"3,626.48","4,038.90","3,670.27","3,396.98","3,965.55",TRAIN_LJUNG_BOX_AND_ROOTS_PASS


**Test forecasts: h = 30 observations (241 rows)**

origin_date,target_date,price_origin,actual,arima_fc,arima_lo95,arima_hi95,diagnostic_status
2025-08-27,2025-10-08,"3,397.42","4,038.90","3,457.21","3,152.37","3,791.53",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-08-28,2025-10-09,"3,417.20","3,976.29","3,477.54","3,170.91","3,813.83",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-08-29,2025-10-10,"3,448.00","4,018.30","3,507.16","3,197.92","3,846.31",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-01,2025-10-13,"3,476.47","4,110.70","3,535.65","3,223.89","3,877.55",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-02,2025-10-14,"3,533.73","4,142.31","3,591.96","3,275.24","3,939.30",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-03,2025-10-15,"3,559.43","4,208.58","3,618.70","3,299.62","3,968.63",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-04,2025-10-16,"3,546.17","4,326.18","3,609.99","3,291.68","3,959.08",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-05,2025-10-17,"3,586.81","4,249.98","3,649.78","3,327.96","4,002.72",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-08,2025-10-20,"3,635.84","4,356.50","3,695.78","3,369.91","4,053.17",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2025-09-09,2025-10-21,"3,626.48","4,125.10","3,690.06","3,364.69","4,046.89",TRAIN_LJUNG_BOX_AND_ROOTS_PASS


### 7.3 Future forecasts from the latest observed date
This seven-row table uses a common origin: the final date in the CSV. Target dates are 1, 3, 5, 7, 14, 21 and 30 observations later.
Only filtering states are updated with the full observed history; model order and train-estimated parameters remain fixed.
These are future values, separate from the historical test forecasts evaluated below. No future actual prices are assumed.

In [12]:
if not accepted_model:
    display(Markdown('**Skipped:** Step 6 accepted no model. No forecast or metric is generated by this cell.'))
else:
    # Update filtering states with the latest observed data, keeping train parameters fixed.
    latest_origin = log_all.index[-1]
    latest_result = res.apply(model_history(log_all), refit=False)
    assert np.array_equal(np.asarray(latest_result.params), frozen_params)
    latest_fc = latest_result.get_forecast(steps=max(max(HORIZONS), max(DAILY_FORECAST_WINDOWS)))
    latest_means = np.exp(np.asarray(latest_fc.predicted_mean))
    latest_ci = np.exp(np.asarray(latest_fc.conf_int(alpha=ALPHA)))
    latest_price = float(df['Price'].iloc[-1])
    future_dates = pd.date_range(latest_origin+calendar_offset, periods=max(max(HORIZONS),max(DAILY_FORECAST_WINDOWS)), freq=calendar_offset)
    future_rows = []
    for h in HORIZONS:
        value = latest_means[h-1]
        if np.isclose(value, latest_price, rtol=32*np.finfo(float).eps, atol=0):
            value = latest_price
        future_rows.append(dict(origin_date=latest_origin, h=h,
            target_date=future_dates[h-1], target_date_status='PROJECTED: source weekday pattern; sessions unknown',
            arima_fc=value, arima_lo95=latest_ci[h-1, 0], arima_hi95=latest_ci[h-1, 1]))
    future_forecasts = pd.DataFrame(future_rows)
    future_forecasts['diagnostic_status'] = diagnostic_status
    assert future_forecasts['h'].tolist() == HORIZONS
    assert (future_forecasts['target_date'] > df.index[-1]).all()
    save_table(future_forecasts, 'future_forecast_by_horizon.csv')
    print(f'Forecast origin: {latest_origin.date()}, last observed price: {latest_price:,.2f} USD')
    display(future_forecasts.style.format({'origin_date': lambda x: x.strftime('%Y-%m-%d'),
        'target_date': lambda x: x.strftime('%Y-%m-%d'),
        'arima_fc': '{:,.2f}', 'arima_lo95': '{:,.2f}', 'arima_hi95': '{:,.2f}'}).hide(axis='index'))


Forecast origin: 2026-09-09, last observed price: 4,401.55 USD


origin_date,h,target_date,target_date_status,arima_fc,arima_lo95,arima_hi95,diagnostic_status
2026-09-09,1,2026-09-10,PROJECTED: source weekday pattern; sessions unknown,"4,406.94","4,328.02","4,487.29",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,3,2026-09-14,PROJECTED: source weekday pattern; sessions unknown,"4,409.25","4,277.33","4,545.24",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,5,2026-09-16,PROJECTED: source weekday pattern; sessions unknown,"4,414.71","4,247.90","4,588.07",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,7,2026-09-18,PROJECTED: source weekday pattern; sessions unknown,"4,419.98","4,224.47","4,624.53",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,14,2026-09-29,PROJECTED: source weekday pattern; sessions unknown,"4,438.50","4,165.91","4,728.91",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,21,2026-10-08,PROJECTED: source weekday pattern; sessions unknown,"4,457.09","4,125.22","4,815.67",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,30,2026-10-21,PROJECTED: source weekday pattern; sessions unknown,"4,481.12","4,086.00","4,914.45",TRAIN_LJUNG_BOX_AND_ROOTS_PASS


### 7.4 Daily future forecast tables  next 7 and 30 observations
 `OUTPUT/ARIMA_BTC/tables/future_forecast_observed-step_7d.csv`  `future_forecast_observed-step_30d.csv`.


In [13]:
if not accepted_model:
    display(Markdown('**Skipped:** Step 6 accepted no model. No forecast or metric is generated by this cell.'))
else:
    # Reuse the same latest-origin forecast path; no model fitting or selection.
    daily_steps = max(DAILY_FORECAST_WINDOWS)
    daily_dates = future_dates[:daily_steps]
    daily_point = latest_means[:daily_steps].copy()
    daily_point[np.isclose(daily_point, latest_price, rtol=32*np.finfo(float).eps, atol=0)] = latest_price
    future_forecast_daily = pd.DataFrame({
        'origin_date': latest_origin,
        'h': np.arange(1, daily_steps + 1),
        'target_date': daily_dates,
        'day_of_week': daily_dates.day_name(),
        'price_origin': latest_price,
        'arima_fc': daily_point,
        'arima_lo95': latest_ci[:daily_steps, 0],
        'arima_hi95': latest_ci[:daily_steps, 1],
    })
    future_forecast_daily['diagnostic_status'] = diagnostic_status
    assert np.array_equal(np.asarray(latest_result.params), frozen_params)
    assert len(future_forecast_daily) == daily_steps
    assert (future_forecast_daily.target_date > latest_origin).all()
    assert future_forecast_daily.h.tolist() == list(range(1,daily_steps+1))
    assert not future_forecast_daily['target_date'].duplicated().any()
    assert np.isfinite(future_forecast_daily.select_dtypes('number').to_numpy()).all()
    assert (future_forecast_daily[['price_origin', 'arima_fc', 'arima_lo95', 'arima_hi95']] > 0).all().all()
    assert (future_forecast_daily['arima_lo95'] <= future_forecast_daily['arima_fc']).all()
    assert (future_forecast_daily['arima_fc'] <= future_forecast_daily['arima_hi95']).all()
    # Requested-horizon rows must agree with section 7.3.
    np.testing.assert_allclose(
        future_forecast_daily.set_index('h').loc[HORIZONS, ['arima_fc', 'arima_lo95', 'arima_hi95']],
        future_forecasts.set_index('h').loc[HORIZONS, ['arima_fc', 'arima_lo95', 'arima_hi95']],
        rtol=1e-12,
    )
    future_daily_tables = {}
    for days in DAILY_FORECAST_WINDOWS:
        table = future_forecast_daily.iloc[:days].copy()
        assert len(table) == days and table['h'].tolist() == list(range(1, days + 1))
        future_daily_tables[days] = table
        save_table(table, f'future_forecast_daily_{days}d.csv')
        display(Markdown(f'**Next {days} projected observations ? origin {latest_origin.date()}, last price {latest_price:,.2f} USD**'))
        shown = table.copy()
        for column in ['origin_date', 'target_date']:
            shown[column] = shown[column].dt.strftime('%Y-%m-%d')
        display(HTML(shown.to_html(index=False, float_format=lambda value: f'{value:,.2f}')))


**Next 7 projected observations ? origin 2026-09-09, last price 4,401.55 USD**

origin_date,h,target_date,day_of_week,price_origin,arima_fc,arima_lo95,arima_hi95,diagnostic_status
2026-09-09,1,2026-09-10,Thursday,"4,401.55","4,406.94","4,328.02","4,487.29",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,2,2026-09-11,Friday,"4,401.55","4,406.76","4,296.30","4,520.06",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,3,2026-09-14,Monday,"4,401.55","4,409.25","4,277.33","4,545.24",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,4,2026-09-15,Tuesday,"4,401.55","4,412.07","4,261.73","4,567.71",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,5,2026-09-16,Wednesday,"4,401.55","4,414.71","4,247.90","4,588.07",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,6,2026-09-17,Thursday,"4,401.55","4,417.34","4,235.59","4,606.88",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,7,2026-09-18,Friday,"4,401.55","4,419.98","4,224.47","4,624.53",TRAIN_LJUNG_BOX_AND_ROOTS_PASS


**Next 30 projected observations ? origin 2026-09-09, last price 4,401.55 USD**

origin_date,h,target_date,day_of_week,price_origin,arima_fc,arima_lo95,arima_hi95,diagnostic_status
2026-09-09,1,2026-09-10,Thursday,"4,401.55","4,406.94","4,328.02","4,487.29",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,2,2026-09-11,Friday,"4,401.55","4,406.76","4,296.30","4,520.06",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,3,2026-09-14,Monday,"4,401.55","4,409.25","4,277.33","4,545.24",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,4,2026-09-15,Tuesday,"4,401.55","4,412.07","4,261.73","4,567.71",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,5,2026-09-16,Wednesday,"4,401.55","4,414.71","4,247.90","4,588.07",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,6,2026-09-17,Thursday,"4,401.55","4,417.34","4,235.59","4,606.88",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,7,2026-09-18,Friday,"4,401.55","4,419.98","4,224.47","4,624.53",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,8,2026-09-21,Monday,"4,401.55","4,422.62","4,214.29","4,641.25",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,9,2026-09-22,Tuesday,"4,401.55","4,425.26","4,204.87","4,657.20",TRAIN_LJUNG_BOX_AND_ROOTS_PASS
2026-09-09,10,2026-09-23,Wednesday,"4,401.55","4,427.90","4,196.11","4,672.51",TRAIN_LJUNG_BOX_AND_ROOTS_PASS


<a id="task1-future-metrics"></a>

### 7.5 — Task 1: Future forecast RMSE/MAPE

**Source:** original saved forecasts from **9 September 2026**, evaluated against `XAU_USD Historical Data (2).csv` in `ACTUAL_FOR_PREDICT`. Match by the original target dates; no retraining, calendar changes, date shifting or price imputation. These future forecasts have now been observed; this evaluation is separate from the historical Test set.

The tables show (A) actual/forecast pairs, (B) requested horizons, and (C) whole 7/30-step paths. **One origin gives n=1 per horizon**, so per-horizon RMSE is absolute error and MAPE is the single percentage error. Path metrics combine different horizons from that origin; they are not rolling-origin Test scores. Missing dates stay N/A and partial-window counts are explicit.

**Units:** RMSE in USD (source price); MAPE in percent. **Forecast precision:** full precision saved forecast CSV; checked against saved notebook output. LSTM/Transformer use the original predefined-seed ensemble forecast, not an average of seed metrics. Archive values are embedded in the next cell and cannot be overwritten by later inference.

[Go to Task 2 — Test set](#task2-test-metrics)


In [2]:
# Evaluate an immutable archive of the original forecasts, without fitting any model.
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML

task1_model = 'ARIMA'
task1_asset = 'XAU'
task1_unit = 'USD (source price)'
task1_horizons = [1, 3, 5, 7, 14, 21, 30]
task1_archive_provenance = {'source': 'OUTPUT\\ARIMA_XAU\\tables\\future_forecast_daily_30d.csv', 'precision': 'full precision saved forecast CSV; checked against saved notebook output', 'original_notebook_sha256': '76b2113f792c9a521dd962f468e731fd1a3f9958280aeff74e2a12928cc7e1f1', 'original_data_source': 'CSV/XAU_USD Historical Data.csv', 'original_data_sha256': '754ad6941968a4f6ca111dbd6e0701a520df9c85a89aa43afe3e36cd9b0a677d'}
task1_archive = pd.DataFrame(json.loads('[{"origin_date": "2026-09-09", "h": 1, "target_date": "2026-09-10", "Forecast": 4406.937917987672}, {"origin_date": "2026-09-09", "h": 2, "target_date": "2026-09-11", "Forecast": 4406.756152556528}, {"origin_date": "2026-09-09", "h": 3, "target_date": "2026-09-14", "Forecast": 4409.250762252757}, {"origin_date": "2026-09-09", "h": 4, "target_date": "2026-09-15", "Forecast": 4412.066891226828}, {"origin_date": "2026-09-09", "h": 5, "target_date": "2026-09-16", "Forecast": 4414.7096566043065}, {"origin_date": "2026-09-09", "h": 6, "target_date": "2026-09-17", "Forecast": 4417.335900583721}, {"origin_date": "2026-09-09", "h": 7, "target_date": "2026-09-18", "Forecast": 4419.975128616364}, {"origin_date": "2026-09-09", "h": 8, "target_date": "2026-09-21", "Forecast": 4422.616937242951}, {"origin_date": "2026-09-09", "h": 9, "target_date": "2026-09-22", "Forecast": 4425.2595824320015}, {"origin_date": "2026-09-09", "h": 10, "target_date": "2026-09-23", "Forecast": 4427.903752484162}, {"origin_date": "2026-09-09", "h": 11, "target_date": "2026-09-24", "Forecast": 4430.549550587549}, {"origin_date": "2026-09-09", "h": 12, "target_date": "2026-09-25", "Forecast": 4433.196932449878}, {"origin_date": "2026-09-09", "h": 13, "target_date": "2026-09-28", "Forecast": 4435.845893090922}, {"origin_date": "2026-09-09", "h": 14, "target_date": "2026-09-29", "Forecast": 4438.496436422294}, {"origin_date": "2026-09-09", "h": 15, "target_date": "2026-09-30", "Forecast": 4441.14856372853}, {"origin_date": "2026-09-09", "h": 16, "target_date": "2026-10-01", "Forecast": 4443.802275762112}, {"origin_date": "2026-09-09", "h": 17, "target_date": "2026-10-02", "Forecast": 4446.457573450918}, {"origin_date": "2026-09-09", "h": 18, "target_date": "2026-10-05", "Forecast": 4449.114457755058}, {"origin_date": "2026-09-09", "h": 19, "target_date": "2026-10-06", "Forecast": 4451.772929623622}, {"origin_date": "2026-09-09", "h": 20, "target_date": "2026-10-07", "Forecast": 4454.432990004405}, {"origin_date": "2026-09-09", "h": 21, "target_date": "2026-10-08", "Forecast": 4457.094639846539}, {"origin_date": "2026-09-09", "h": 22, "target_date": "2026-10-09", "Forecast": 4459.757880099822}, {"origin_date": "2026-09-09", "h": 23, "target_date": "2026-10-12", "Forecast": 4462.422711714567}, {"origin_date": "2026-09-09", "h": 24, "target_date": "2026-10-13", "Forecast": 4465.089135641658}, {"origin_date": "2026-09-09", "h": 25, "target_date": "2026-10-14", "Forecast": 4467.757152832555}, {"origin_date": "2026-09-09", "h": 26, "target_date": "2026-10-15", "Forecast": 4470.426764239263}, {"origin_date": "2026-09-09", "h": 27, "target_date": "2026-10-16", "Forecast": 4473.09797081438}, {"origin_date": "2026-09-09", "h": 28, "target_date": "2026-10-19", "Forecast": 4475.770773511069}, {"origin_date": "2026-09-09", "h": 29, "target_date": "2026-10-20", "Forecast": 4478.445173283044}, {"origin_date": "2026-09-09", "h": 30, "target_date": "2026-10-21", "Forecast": 4481.12117108461}]'))
task1_root = next(base for base in [Path.cwd(), *Path.cwd().parents]
    if (base / 'ACTUAL_FOR_PREDICT').is_dir() and (base / 'CSV').is_dir())
task1_actual_path = task1_root / 'ACTUAL_FOR_PREDICT' / 'XAU_USD Historical Data (2).csv'
task1_actual = pd.read_csv(task1_actual_path, encoding='utf-8-sig')
task1_actual['target_date'] = pd.to_datetime(task1_actual['Date'], format='%m/%d/%Y', errors='raise')
task1_actual['Actual'] = pd.to_numeric(task1_actual['Price'].astype(str).str.replace(',', '', regex=False), errors='raise')
assert task1_actual['target_date'].notna().all() and not task1_actual['target_date'].duplicated().any()
assert np.isfinite(task1_actual['Actual']).all() and task1_actual['Actual'].gt(0).all()
for task1_column in ['origin_date', 'target_date']:
    task1_archive[task1_column] = pd.to_datetime(task1_archive[task1_column], errors='raise')
assert task1_archive['h'].tolist() == list(range(1, 31))
assert task1_archive['origin_date'].nunique() == 1 and task1_archive['target_date'].is_unique
assert task1_archive['target_date'].is_monotonic_increasing
assert (task1_archive['target_date'] > task1_archive['origin_date']).all()
assert np.isfinite(task1_archive['Forecast']).all() and task1_archive['Forecast'].gt(0).all()

task1_comparison = task1_archive.merge(task1_actual[['target_date', 'Actual']],
    on='target_date', how='left', validate='one_to_one')
np.testing.assert_array_equal(task1_comparison['Forecast'], task1_archive['Forecast'])
assert len(task1_comparison) == 30
task1_comparison['Error'] = task1_comparison['Forecast'] - task1_comparison['Actual']
task1_comparison['APE (%)'] = 100 * task1_comparison['Error'].abs() / task1_comparison['Actual']
task1_comparison['Status'] = np.where(task1_comparison['Actual'].notna(),
    'Matched original target date', 'Missing actual: original target date retained')
task1_horizon_metrics = task1_comparison.loc[task1_comparison.h.isin(task1_horizons),
    ['origin_date', 'h', 'target_date', 'Forecast', 'Actual']].copy()
task1_horizon_metrics['n_actuals'] = task1_horizon_metrics['Actual'].notna().astype(int)
task1_horizon_metrics['RMSE'] = (task1_horizon_metrics['Forecast'] - task1_horizon_metrics['Actual']).abs()
task1_horizon_metrics['RMSE_unit'] = task1_unit
task1_horizon_metrics['MAPE (%)'] = 100 * task1_horizon_metrics['RMSE'] / task1_horizon_metrics['Actual']
task1_window_rows = []
for task1_window in [7, 30]:
    task1_part = task1_comparison.loc[task1_comparison.h.le(task1_window)]
    task1_matched = task1_part.dropna(subset=['Actual'])
    task1_error = task1_matched['Forecast'].to_numpy() - task1_matched['Actual'].to_numpy()
    task1_window_rows.append(dict(window_steps=task1_window, n_expected=len(task1_part),
        n_actuals=len(task1_matched), n_missing=len(task1_part)-len(task1_matched),
        first_target_date=task1_part.target_date.min(), last_target_date=task1_part.target_date.max(),
        RMSE=float(np.sqrt(np.mean(task1_error**2))) if len(task1_matched) else np.nan,
        RMSE_unit=task1_unit,
        MAPE_percent=float(100*np.mean(np.abs(task1_error/task1_matched['Actual'].to_numpy()))) if len(task1_matched) else np.nan,
        Status='Complete' if len(task1_part)==len(task1_matched) else 'Partial: matched dates only'))
task1_window_metrics = pd.DataFrame(task1_window_rows).rename(columns={'MAPE_percent': 'MAPE (%)'})
task1_missing = task1_comparison.loc[task1_comparison.Actual.isna(), ['h', 'target_date']].copy()

def task1_show_table(frame):
    shown = frame.copy()
    for column in ['origin_date', 'target_date', 'first_target_date', 'last_target_date']:
        if column in shown:
            shown[column] = pd.to_datetime(shown[column]).dt.strftime('%Y-%m-%d')
    display(HTML('<div style="max-height:420px;overflow:auto;border:1px solid #ddd;padding:8px">'
        + shown.to_html(index=False, na_rep='N/A', float_format=lambda value: f'{value:,.6f}') + '</div>'))

display(Markdown('**Task 1A — Original future forecast / actual pairs (prices in ' + task1_unit + ')**'))
task1_show_table(task1_comparison)
display(Markdown('**Task 1B — Future RMSE/MAPE by horizon; one original origin, n=1 per horizon**'))
task1_show_table(task1_horizon_metrics)
display(Markdown('**Task 1C — RMSE/MAPE across each plotted 7/30-step future path (mixed horizons)**'))
task1_show_table(task1_window_metrics)
if not task1_missing.empty:
    display(Markdown('**Missing actual prices: no date shifting or imputation**'))
    task1_show_table(task1_missing)
task1_output = task1_root / 'OUTPUT' / 'METRICS_REVIEW' / 'Task1_Future' / 'ARIMA_XAU'
task1_output.mkdir(parents=True, exist_ok=True)
for task1_name, task1_frame in [('original_forecast_archive.csv', task1_archive),
        ('forecast_actual_pairs.csv', task1_comparison), ('metrics_by_horizon.csv', task1_horizon_metrics),
        ('metrics_7_30_steps.csv', task1_window_metrics), ('missing_actual_dates.csv', task1_missing)]:
    task1_frame.to_csv(task1_output / task1_name, index=False, encoding='utf-8-sig')
task1_provenance = dict(task1_archive_provenance, actual_source=str(task1_actual_path.relative_to(task1_root)),
    actual_sha256=hashlib.sha256(task1_actual_path.read_bytes()).hexdigest(),
    model=task1_model, asset=task1_asset, origin=str(task1_archive.origin_date.iloc[0].date()),
    forecast_sha256=hashlib.sha256(task1_archive.to_csv(index=False).encode('utf-8')).hexdigest(),
    matching='exact original target date; no imputation or date shifts', refit_for_evaluation=False)
(task1_output / 'provenance.json').write_text(json.dumps(task1_provenance, indent=2, ensure_ascii=False), encoding='utf-8')
print('Forecast source:', task1_archive_provenance['source'])
print('Actual prices:', task1_actual_path.name, '| forecast origin:', task1_archive.origin_date.iloc[0].date())
print('Per-horizon n=1: RMSE equals absolute error; MAPE is the single absolute percentage error.')
print('Window metrics mix horizons from one origin and are separate from Test-set performance.')
print('CSV results:', task1_output)


**Task 1A — Original future forecast / actual pairs (prices in USD (source price))**

origin_date,h,target_date,Forecast,Actual,Error,APE (%),Status
2026-09-09,1,2026-09-10,"4,406.937918","4,317.300000",89.637918,2.076249,Matched original target date
2026-09-09,2,2026-09-11,"4,406.756153","4,348.720000",58.036153,1.334557,Matched original target date
2026-09-09,3,2026-09-14,"4,409.250762","4,298.800000",110.450762,2.569339,Matched original target date
2026-09-09,4,2026-09-15,"4,412.066891","4,294.000000",118.066891,2.749578,Matched original target date
2026-09-09,5,2026-09-16,"4,414.709657","4,263.500000",151.209657,3.546609,Matched original target date
2026-09-09,6,2026-09-17,"4,417.335901","4,341.300000",76.035901,1.751455,Matched original target date
2026-09-09,7,2026-09-18,"4,419.975129","4,378.170000",41.805129,0.954854,Matched original target date
2026-09-09,8,2026-09-21,"4,422.616937","4,343.560000",79.056937,1.820095,Matched original target date
2026-09-09,9,2026-09-22,"4,425.259582","4,355.480000",69.779582,1.602110,Matched original target date
2026-09-09,10,2026-09-23,"4,427.903752","4,287.510000",140.393752,3.274482,Matched original target date


**Task 1B — Future RMSE/MAPE by horizon; one original origin, n=1 per horizon**

origin_date,h,target_date,Forecast,Actual,n_actuals,RMSE,RMSE_unit,MAPE (%)
2026-09-09,1,2026-09-10,"4,406.937918","4,317.300000",1,89.637918,USD (source price),2.076249
2026-09-09,3,2026-09-14,"4,409.250762","4,298.800000",1,110.450762,USD (source price),2.569339
2026-09-09,5,2026-09-16,"4,414.709657","4,263.500000",1,151.209657,USD (source price),3.546609
2026-09-09,7,2026-09-18,"4,419.975129","4,378.170000",1,41.805129,USD (source price),0.954854
2026-09-09,14,2026-09-29,"4,438.496436","4,182.800000",1,255.696436,USD (source price),6.113045
2026-09-09,21,2026-10-08,"4,457.094640","4,134.930000",1,322.164640,USD (source price),7.791296
2026-09-09,30,2026-10-21,"4,481.121171","4,140.970000",1,340.151171,USD (source price),8.214287


**Task 1C — RMSE/MAPE across each plotted 7/30-step future path (mixed horizons)**

window_steps,n_expected,n_actuals,n_missing,first_target_date,last_target_date,RMSE,RMSE_unit,MAPE (%),Status
7,7,7,0,2026-09-10,2026-09-18,98.514539,USD (source price),2.140377,Complete
30,30,30,0,2026-09-10,2026-10-21,249.774626,USD (source price),5.416178,Complete


Forecast source: OUTPUT\ARIMA_XAU\tables\future_forecast_daily_30d.csv
Actual prices: XAU_USD Historical Data (2).csv | forecast origin: 2026-09-09
Per-horizon n=1: RMSE equals absolute error; MAPE is the single absolute percentage error.
Window metrics mix horizons from one origin and are separate from Test-set performance.
CSV results: C:\Users\User\Desktop\PROJECT_2026\OUTPUT\METRICS_REVIEW\Task1_Future\ARIMA_XAU


## Step 8 - RMSE and MAPE on Test
**RMSE (USD)** = sqrt(mean((actual - forecast)^2)). **MAPE (%)** = 100 * mean(abs((actual - forecast) / actual)).
Both are computed separately for each horizon using only target dates in test. Lower values indicate smaller errors.
The actual price is needed to calculate errors. Future forecasts in Section 7.3 are not included.

### 8.1 Metrics by horizon

In [14]:
if not accepted_model:
    display(Markdown('**Skipped:** Step 6 accepted no model. No forecast or metric is generated by this cell.'))
else:
    evaluation_rows = []
    for h, group in forecasts.groupby('h', sort=True):
        actual = group['actual'].to_numpy()
        predicted = group['arima_fc'].to_numpy()
        assert np.isfinite(actual).all() and np.isfinite(predicted).all() and (actual != 0).all()
        error = actual - predicted
        evaluation_rows.append(dict(h=h, n_points=len(group),
            RMSE=np.sqrt(np.mean(error**2)), MAPE=100*np.mean(np.abs(error / actual))))
    evaluation = pd.DataFrame(evaluation_rows).sort_values('h').reset_index(drop=True)
    assert len(evaluation) == len(HORIZONS)
    evaluation['diagnostic_status'] = diagnostic_status
    save_table(evaluation, 'evaluation_by_horizon.csv')
    display(evaluation.style.format({'RMSE': '{:,.2f}', 'MAPE': '{:.2f}'}).hide(axis='index'))


h,n_points,RMSE,MAPE,diagnostic_status
1,270,80.43,1.26,TRAIN_LJUNG_BOX_AND_ROOTS_PASS
3,268,136.23,2.20,TRAIN_LJUNG_BOX_AND_ROOTS_PASS
5,266,173.11,2.99,TRAIN_LJUNG_BOX_AND_ROOTS_PASS
7,264,203.99,3.53,TRAIN_LJUNG_BOX_AND_ROOTS_PASS
14,257,276.79,4.82,TRAIN_LJUNG_BOX_AND_ROOTS_PASS
21,250,330.92,6.11,TRAIN_LJUNG_BOX_AND_ROOTS_PASS
30,241,400.95,7.82,TRAIN_LJUNG_BOX_AND_ROOTS_PASS


<a id="task2-test-metrics"></a>

### Task 2: Test set RMSE/MAPE

**Original location:** Step 8 - RMSE and MAPE on Test. This table recomputes scores from the original chronological **Test 20% only**, grouped by forecast horizon. Future-actual files in `ACTUAL_FOR_PREDICT` are excluded. Dates, sample counts and actual prices are checked against the unchanged original dataset; no model is fitted in this cell.

**Forecast kind:** model point forecast. RMSE is in USD (source price); MAPE is percent. Seed means/SD, when present above, are separate from ensemble performance. The saved report uses `OUTPUT\ARIMA_XAU\tables\forecasts_long.csv (full precision; matched saved notebook tables)`. Rerunning the full notebook uses its in-memory Test forecasts at full precision.

[Go to Task 1 — Future forecast](#task1-future-metrics)


In [3]:
# Recompute Test-set metrics from the original historical Test forecasts only.
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display, HTML

task2_primary_column = 'arima_fc'
task2_unit = 'USD (source price)'
task2_model = 'ARIMA'
task2_asset = 'XAU'
task2_kind = 'model point forecast'
task2_test_pairs = forecasts[['origin_date', 'h', 'target_date', 'actual', task2_primary_column]].copy()
task2_test_pairs = task2_test_pairs.rename(columns={task2_primary_column: 'Forecast'})
for task2_column in ['origin_date', 'target_date']:
    task2_test_pairs[task2_column] = pd.to_datetime(task2_test_pairs[task2_column], errors='raise')
assert n_train == int(0.8 * len(df))
assert task2_test_pairs['target_date'].between(df.index[n_train], df.index[-1]).all()
assert not task2_test_pairs.duplicated(['origin_date', 'h']).any()
assert np.isfinite(task2_test_pairs[['actual', 'Forecast']]).all().all()
task2_target_positions = df.index.get_indexer(task2_test_pairs.target_date)
task2_origin_positions = df.index.get_indexer(task2_test_pairs.origin_date)
assert (task2_origin_positions >= 0).all() and (task2_target_positions >= n_train).all()
assert np.array_equal(task2_target_positions-task2_origin_positions, task2_test_pairs.h.to_numpy())
np.testing.assert_allclose(task2_test_pairs.actual.to_numpy(), df['Price'].iloc[task2_target_positions].to_numpy(), atol=0, rtol=1e-12)
task2_rows = []
for task2_h, task2_group in task2_test_pairs.groupby('h', sort=True):
    assert len(task2_group) == len(df)-n_train-int(task2_h)+1
    task2_actual = task2_group.actual.to_numpy()
    task2_error = task2_group.Forecast.to_numpy()-task2_actual
    task2_zero_count = int((task2_actual == 0).sum())
    task2_rows.append(dict(partition='Test', h=int(task2_h), n_points=len(task2_group),
        first_target_date=task2_group.target_date.min(), last_target_date=task2_group.target_date.max(),
        RMSE=float(np.sqrt(np.mean(task2_error**2))), RMSE_unit=task2_unit,
        MAPE_percent=float(100*np.mean(np.abs(task2_error/task2_actual))) if not task2_zero_count else np.nan,
        zero_actuals=task2_zero_count, forecast_kind=task2_kind))
task2_test_metrics = pd.DataFrame(task2_rows).rename(columns={'MAPE_percent': 'MAPE (%)'})
assert task2_test_metrics.h.tolist() == sorted(HORIZONS)
task2_shown = task2_test_metrics.copy()
for task2_column in ['first_target_date', 'last_target_date']:
    task2_shown[task2_column] = task2_shown[task2_column].dt.strftime('%Y-%m-%d')
display(HTML('<div style="max-height:420px;overflow:auto;border:1px solid #ddd;padding:8px">'
    + task2_shown.to_html(index=False, na_rep='N/A', float_format=lambda value: f'{value:,.6f}') + '</div>'))
task2_root = next(base for base in [Path.cwd(), *Path.cwd().parents]
    if (base / 'ACTUAL_FOR_PREDICT').is_dir() and (base / 'CSV').is_dir())
task2_output = task2_root / 'OUTPUT' / 'METRICS_REVIEW' / 'Task2_Test' / 'ARIMA_XAU'
task2_output.mkdir(parents=True, exist_ok=True)
task2_test_metrics.to_csv(task2_output / 'metrics_by_horizon.csv', index=False, encoding='utf-8-sig')
task2_test_pairs.to_csv(task2_output / 'test_forecast_pairs.csv', index=False, encoding='utf-8-sig')
print('Partition: original chronological Test 20%; actuals from ACTUAL_FOR_PREDICT are excluded.')
print('Test source:', 'OUTPUT\\ARIMA_XAU\\tables\\forecasts_long.csv (full precision; matched saved notebook tables)')
print('Forecast kind:', task2_kind, '| Units:', task2_unit, '| MAPE: percent')
print('CSV results:', task2_output)


partition,h,n_points,first_target_date,last_target_date,RMSE,RMSE_unit,MAPE (%),zero_actuals,forecast_kind
Test,1,270,2025-08-28,2026-09-09,80.429113,USD (source price),1.261195,0,model point forecast
Test,3,268,2025-09-01,2026-09-09,136.232565,USD (source price),2.201919,0,model point forecast
Test,5,266,2025-09-03,2026-09-09,173.112700,USD (source price),2.987060,0,model point forecast
Test,7,264,2025-09-05,2026-09-09,203.994543,USD (source price),3.532797,0,model point forecast
Test,14,257,2025-09-16,2026-09-09,276.785799,USD (source price),4.823251,0,model point forecast
Test,21,250,2025-09-25,2026-09-09,330.922363,USD (source price),6.105725,0,model point forecast
Test,30,241,2025-10-08,2026-09-09,400.946838,USD (source price),7.817407,0,model point forecast


Partition: original chronological Test 20%; actuals from ACTUAL_FOR_PREDICT are excluded.
Test source: OUTPUT\ARIMA_XAU\tables\forecasts_long.csv (full precision; matched saved notebook tables)
Forecast kind: model point forecast | Units: USD (source price) | MAPE: percent
CSV results: C:\Users\User\Desktop\PROJECT_2026\OUTPUT\METRICS_REVIEW\Task2_Test\ARIMA_XAU


### 8.2 Actual and forecast plots for each horizon

In [15]:
if not accepted_model:
    display(Markdown('**Skipped:** Step 6 accepted no model. No forecast or metric is generated by this cell.'))
else:
    for h, group in forecasts.groupby('h', sort=True):
        fig, ax = plt.subplots(figsize=(13, 5))
        ax.plot(group['target_date'], group['actual'], color='black', label='Actual', linewidth=1.4)
        ax.plot(group['target_date'], group['arima_fc'], label=f'ARIMA h={h}', linewidth=1)
        ax.fill_between(group['target_date'], group['arima_lo95'], group['arima_hi95'], alpha=0.13, label='ARIMA 95% PI')
        ax.set(title=f'Rolling-origin forecasts — {h} observations ahead', xlabel='Target date', ylabel='Price (USD)')
        ax.legend()
        save_figure(fig, f'09_forecast_h{h}.png')
    


C:\Users\User\AppData\Local\Temp\ipykernel_22704\1610437506.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.3 Static 30-observation forecast from the final train origin

In [16]:
if not accepted_model:
    display(Markdown('**Skipped:** Step 6 accepted no model. No forecast or metric is generated by this cell.'))
else:
    static_fc = res.get_forecast(steps=30)
    static_dates = test_price.index[:30]
    static_mean = np.exp(np.asarray(static_fc.predicted_mean))
    static_ci = np.exp(np.asarray(static_fc.conf_int(alpha=ALPHA)))
    save_table(pd.DataFrame(dict(target_date=static_dates, diagnostic_status=diagnostic_status, arima_fc=static_mean,
        arima_lo95=static_ci[:, 0], arima_hi95=static_ci[:, 1])), 'static_30d_forecast.csv')
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(train_price.iloc[-90:], color='black', label='Train history (last 90 observations)')
    ax.plot(static_dates, static_mean, label='30-observation forecast from final train origin')
    ax.fill_between(static_dates, static_ci[:, 0], static_ci[:, 1], alpha=0.2, label='95% PI')
    ax.plot(test_price.iloc[:30], linestyle='--', color='tab:orange', label='Actual test (evaluation)')
    ax.axvline(train_price.index[-1], color='gray', linestyle=':')
    ax.set(title=f'Static 30-observation forecast — origin {train_price.index[-1].date()}', ylabel='Price (USD)', xlabel='Date')
    ax.legend()
    save_figure(fig, '10_static_30d_fan.png')
    


C:\Users\User\AppData\Local\Temp\ipykernel_22704\1610437506.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 8.4 RMSE and MAPE versus horizon

In [17]:
if not accepted_model:
    display(Markdown('**Skipped:** Step 6 accepted no model. No forecast or metric is generated by this cell.'))
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, metric, unit in zip(axes, ['RMSE', 'MAPE'], ['USD', '%']):
        ax.plot(evaluation['h'], evaluation[metric], marker='o', label='ARIMA')
        ax.set(title=metric, xlabel='Horizon (observations)', ylabel=f'{metric} ({unit})', xticks=HORIZONS)
        ax.legend()
    save_figure(fig, '11_error_vs_horizon.png')


C:\Users\User\AppData\Local\Temp\ipykernel_22704\1610437506.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 9 - Summary and Interpretation
Summarize the selected model, train-only diagnostic evidence, test RMSE and MAPE by horizon, and the future forecast values.
Test metrics measure historical holdout performance. Future forecast rows have no observed target prices yet and are not included in those metrics.


In [18]:
# Summarize this execution, including the legitimate no-model outcome.
assert np.isfinite(df.Price).all() and df.Price.gt(0).all()
assert n_train == int(.8*n) and n_train+n_test == n
required_train_tables = ['library_versions.csv','data_quality.csv','split_summary.csv','stationarity_tests.csv',
    'grid_search_aic_bic.csv','top3_candidates.csv','auto_arima_cross_check.csv',
    'assumption_checks.csv','parameter_significance.csv','model_roots.csv','candidate_diagnostics.csv']
assert all(name in notebook_tables for name in required_train_tables)
lines = ['# XAU ARIMA - current execution', '',
    'Implementation reference: BTC_TEST/ARIMA_BTC.ipynb; Step 1-9 layout and strict AIC Top-3 Ljung-Box/root selection retained.',
    f'Data: {n} observed prices; Train/Test: {n_train}/{n_test}. Train ends {train_log.index[-1].date()}.',
    f'Units: {UNIT}. {CALENDAR_NOTE}',
    'No validation, no Test-guided selection, no full-data refit.',
    '', '## Selection status',
    (f'Forecast model: {final_label}; mode={selection_mode}; status={diagnostic_status}' if accepted_model else 'NO ACCEPTED MODEL: no Top 3 fit passes both Ljung-Box and roots.'),
    ('Train Ljung-Box and roots passed.' if train_diagnostics_passed else 'No candidate meets both required Train gates; no forecasts are produced.'),
    '', '## Train stationarity', stationarity_tests.to_string(index=False),
    '', '## Top 3 diagnostics', candidate_diagnostics.to_string(index=False),
    '', '## Source audit',
    f'Rows with missing auxiliary volume: {df.Volume.isna().sum()}; source OHLC flags: {ohlc_flag.sum()}. Target Price is retained unchanged.',
    'Only log Price is modeled. OHLC flags are disclosed source issues, not a demonstrated cause of residual autocorrelation.',
    'A lower AIC does not guarantee white-noise residuals. Passing a diagnostic is not proof of forecast accuracy.',
    'Ljung-Box alpha=0.05 and lags 10/20/30 match BTC. A failed check remains FAIL; no fallback bypasses it.']
if accepted_model:
    required_forecast_tables = ['forecasts_long.csv','evaluation_by_horizon.csv','future_forecast_by_horizon.csv','static_30d_forecast.csv',
        *[f'forecasts_h{h}.csv' for h in HORIZONS], *[f'future_forecast_daily_{steps}d.csv' for steps in DAILY_FORECAST_WINDOWS]]
    assert all(name in notebook_tables for name in required_forecast_tables)
    assert np.array_equal(np.asarray(res.params), frozen_params)
    assert counts.to_dict() == {h:n_test-h+1 for h in HORIZONS}
    lines += ['', '## Test RMSE/MAPE', evaluation.to_string(index=False),
        '', '## Future forecasts', future_forecasts.to_string(index=False),
        'Point forecasts are exponentiated log-scale medians; intervals are model-based. ARCH/non-normality can impair coverage.']
else:
    lines += ['', '## Why no forecast appears',
        'No converged Top 3 candidate passes both required Train Ljung-Box and root gates.',
        'Steps 7-8 were deliberately skipped. No Test/future forecasts, RMSE or MAPE were calculated in this execution.',
        '**Any old forecast/model files in OUTPUT/ARIMA_XAU are from earlier runs and are not current results.**']
lines += ['', '## Limits',
    'The source has observed weekdays and date gaps. Bitcoin daily-calendar assumptions were not imposed on XAU; no missing prices were filled.',
    'Venue, fixing, adjustments, physical units and session timezone are not supplied. Future dates are weekday-pattern projections, not verified holiday sessions.',
    'This user-requested redesign follows prior Test inspection; the existing Test period is not a new untouched holdout.',
    'Robust covariance is reporting-only and does not fix volatility dynamics or forecast interval calibration.',
    'Univariate prices do not observe news, policy, interest rates, inflation or unexpected events.']
summary_markdown = '\n\n'.join(lines)
display(Markdown(summary_markdown))
(SUMMARY_DIR/'execution_summary.md').write_text(summary_markdown, encoding='utf-8')
(SUMMARY_DIR/'summary.md').write_text(summary_markdown, encoding='utf-8')
display(Markdown('### Current-run diagnostic tables'))
for name in ['top3_candidates.csv','candidate_diagnostics.csv','assumption_checks.csv','source_ohlc_flags.csv']:
    display(Markdown(f'**{name}**'))
    show_table(notebook_tables[name])
audit = dict(completed=accepted_model, notebook_execution_completed=True,
    status='COMPLETED' if accepted_model else 'NO_ACCEPTED_MODEL', asset=ASSET,
    selection_mode=selection_mode, train_diagnostics_passed=train_diagnostics_passed, diagnostic_status=diagnostic_status,
    model_accepted=accepted_model, selected_model=final_label, forecasts_generated=accepted_model,
    steps_executed=list(range(1,10)), forecast_steps_skipped=[] if accepted_model else [7,8],
    n=n, n_train=n_train, n_test=n_test, horizons=HORIZONS,
    selection=configuration['selection'], reference_sha256=configuration['reference_sha256'],
    source_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    candidate_diagnostics=candidate_diagnostics.to_dict('records'),
    python_executable=sys.executable, table_count=len(notebook_tables), figure_count=len(notebook_figures),
    prior_exports_current=accepted_model)
if accepted_model:
    audit.update(parameters_frozen=np.array_equal(np.asarray(res.params), frozen_params),
        counts={int(h):int(c) for h,c in forecasts.groupby('h').size().items()})
write_json(SUMMARY_DIR/'execution_audit.json', audit)
print('Run All completed. Forecasts generated:', accepted_model, '; diagnostic status:', diagnostic_status)


# XAU ARIMA - current execution



Implementation reference: BTC_TEST/ARIMA_BTC.ipynb; Step 1-9 layout and strict AIC Top-3 Ljung-Box/root selection retained.

Data: 1350 observed prices; Train/Test: 1080/270. Train ends 2025-08-27.

Units: USD (source price). Horizons count successive observed records. Future dates use the source weekday pattern, not confirmed exchange sessions; source holidays/omissions and instrument details are unknown.

No validation, no Test-guided selection, no full-data refit.



## Selection status

Forecast model: ARIMA(2,1,0), trend=t; mode=STRICT_DIAGNOSTICS; status=TRAIN_LJUNG_BOX_AND_ROOTS_PASS

Train Ljung-Box and roots passed.



## Train stationarity

        series test  statistic  p_value                                                                  critical_values                         conclusion  stationary                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       warnings
     log_price  ADF   0.898343 0.993069 {"1%": -3.436436308605049, "5%": -2.8642273405849363, "10%": -2.568200833956216} non-stationary (fail to reject H0)       False                                                                                                                                                                                          adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
     log_price KPSS   4.575717 0.010000                          {"10%": 0.347, "5%": 0.463, "2.5%": 0.574, "1%": 0.739}                     non-stationary       False The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is smaller than the p-value returned. | kpss currently returns a plain tuple whose length and layout depends on the store argument (and which silently drops `lags` when store=True). In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning a KPSSResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
diff_log_price  ADF -24.854400 0.000000 {"1%": -3.436436308605049, "5%": -2.8642273405849363, "10%": -2.568200833956216}                         stationary        True                                                                                                                                                                                          adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
diff_log_price KPSS   0.308871 0.100000                          {"10%": 0.347, "5%": 0.463, "2.5%": 0.574, "1%": 0.739}     stationary (fail to reject H0)        True The test statistic is outside of the range of p-values available in the look-up table. The actual p-value is greater than the p-value returned. | kpss currently returns a plain tuple whose length and layout depends on the store argument (and which silently drops `lags` when store=True). In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning a KPSSResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.



## Top 3 diagnostics

                model          AIC          BIC  ljung_box_pass  roots_pass  arch_pvalue  lb_pvalue_10  lb_pvalue_20  lb_pvalue_30  selected     selection_mode
ARIMA(2,1,0), trend=t -7042.017943 -7022.082783            True        True 3.760903e-11      0.928211      0.564085      0.375138      True STRICT_DIAGNOSTICS
ARIMA(0,1,2), trend=t -7041.776032 -7021.840872            True        True 4.282623e-11      0.912813      0.542422      0.362173     False STRICT_DIAGNOSTICS
ARIMA(0,1,0), trend=t -7041.466921 -7031.499341            True        True 1.885055e-10      0.678470      0.356224      0.318482     False STRICT_DIAGNOSTICS



## Source audit

Rows with missing auxiliary volume: 1350; source OHLC flags: 11. Target Price is retained unchanged.

Only log Price is modeled. OHLC flags are disclosed source issues, not a demonstrated cause of residual autocorrelation.

A lower AIC does not guarantee white-noise residuals. Passing a diagnostic is not proof of forecast accuracy.

Ljung-Box alpha=0.05 and lags 10/20/30 match BTC. A failed check remains FAIL; no fallback bypasses it.



## Test RMSE/MAPE

 h  n_points       RMSE     MAPE              diagnostic_status
 1       270  80.429113 1.261195 TRAIN_LJUNG_BOX_AND_ROOTS_PASS
 3       268 136.232565 2.201919 TRAIN_LJUNG_BOX_AND_ROOTS_PASS
 5       266 173.112700 2.987060 TRAIN_LJUNG_BOX_AND_ROOTS_PASS
 7       264 203.994543 3.532797 TRAIN_LJUNG_BOX_AND_ROOTS_PASS
14       257 276.785799 4.823251 TRAIN_LJUNG_BOX_AND_ROOTS_PASS
21       250 330.922363 6.105725 TRAIN_LJUNG_BOX_AND_ROOTS_PASS
30       241 400.946838 7.817407 TRAIN_LJUNG_BOX_AND_ROOTS_PASS



## Future forecasts

origin_date  h target_date                                  target_date_status    arima_fc  arima_lo95  arima_hi95              diagnostic_status
 2026-09-09  1  2026-09-10 PROJECTED: source weekday pattern; sessions unknown 4406.937918 4328.020250 4487.294581 TRAIN_LJUNG_BOX_AND_ROOTS_PASS
 2026-09-09  3  2026-09-14 PROJECTED: source weekday pattern; sessions unknown 4409.250762 4277.331649 4545.238452 TRAIN_LJUNG_BOX_AND_ROOTS_PASS
 2026-09-09  5  2026-09-16 PROJECTED: source weekday pattern; sessions unknown 4414.709657 4247.896672 4588.073312 TRAIN_LJUNG_BOX_AND_ROOTS_PASS
 2026-09-09  7  2026-09-18 PROJECTED: source weekday pattern; sessions unknown 4419.975129 4224.468496 4624.529727 TRAIN_LJUNG_BOX_AND_ROOTS_PASS
 2026-09-09 14  2026-09-29 PROJECTED: source weekday pattern; sessions unknown 4438.496436 4165.913943 4728.914443 TRAIN_LJUNG_BOX_AND_ROOTS_PASS
 2026-09-09 21  2026-10-08 PROJECTED: source weekday pattern; sessions unknown 4457.094640 4125.219456 4815.669285 TRAIN_LJUNG_BOX_AND_ROOTS_PASS
 2026-09-09 30  2026-10-21 PROJECTED: source weekday pattern; sessions unknown 4481.121171 4085.998587 4914.452739 TRAIN_LJUNG_BOX_AND_ROOTS_PASS

Point forecasts are exponentiated log-scale medians; intervals are model-based. ARCH/non-normality can impair coverage.



## Limits

The source has observed weekdays and date gaps. Bitcoin daily-calendar assumptions were not imposed on XAU; no missing prices were filled.

Venue, fixing, adjustments, physical units and session timezone are not supplied. Future dates are weekday-pattern projections, not verified holiday sessions.

This user-requested redesign follows prior Test inspection; the existing Test period is not a new untouched holdout.

Robust covariance is reporting-only and does not fix volatility dynamics or forecast interval calibration.

Univariate prices do not observe news, policy, interest rates, inflation or unexpected events.

### Current-run diagnostic tables

**top3_candidates.csv**

AIC Rank,model,p,d,q,trend,AIC,BIC,converged,warnings,error
1,"ARIMA(2,1,0), trend=t",2,1,0,t,-7042.017943,-7022.082783,True,,
2,"ARIMA(0,1,2), trend=t",0,1,2,t,-7041.776032,-7021.840872,True,,
3,"ARIMA(0,1,0), trend=t",0,1,0,t,-7041.466921,-7031.499341,True,,


**candidate_diagnostics.csv**

model,AIC,BIC,ljung_box_pass,roots_pass,arch_pvalue,lb_pvalue_10,lb_pvalue_20,lb_pvalue_30,selected,selection_mode
"ARIMA(2,1,0), trend=t",-7042.017943,-7022.082783,True,True,0.000000,0.928211,0.564085,0.375138,True,STRICT_DIAGNOSTICS
"ARIMA(0,1,2), trend=t",-7041.776032,-7021.840872,True,True,0.000000,0.912813,0.542422,0.362173,False,STRICT_DIAGNOSTICS
"ARIMA(0,1,0), trend=t",-7041.466921,-7031.499341,True,True,0.000000,0.678470,0.356224,0.318482,False,STRICT_DIAGNOSTICS


**assumption_checks.csv**

model,test,statistic,p_value,pass_fail,note
"ARIMA(2,1,0), trend=t",Ljung-Box lag 10,3.095667,0.928211,PASS,model_df=2
"ARIMA(2,1,0), trend=t",Ljung-Box lag 20,16.408158,0.564085,PASS,model_df=2
"ARIMA(2,1,0), trend=t",Ljung-Box lag 30,29.752650,0.375138,PASS,model_df=2
"ARIMA(2,1,0), trend=t",Mean = 0 (t-test),0.011710,0.990659,PASS,Residual mean=3.29012e-06
"ARIMA(2,1,0), trend=t",ARCH LM (10 lags),70.370106,0.000000,FAIL,
"ARIMA(2,1,0), trend=t",Jarque-Bera,78.259241,0.000000,FAIL,"skew=0.0182, kurtosis=4.3189"
"ARIMA(2,1,0), trend=t",AR roots outside unit circle,3.960369,NaN,PASS,All |root|>1: True
"ARIMA(2,1,0), trend=t",MA roots outside unit circle,NaN,NaN,REPORTED,N/A: no roots
"ARIMA(2,1,0), trend=t",Parameter significance: x1,0.000597,0.023511,PASS,"Report only; estimate in statistic column, significance uses p < alpha"
"ARIMA(2,1,0), trend=t",Parameter significance: ar.L1,-0.013294,0.617980,FAIL,"Report only; estimate in statistic column, significance uses p < alpha"


**source_ohlc_flags.csv**

Date,Price,Open,High,Low,Vol.,Change %,Volume
2022-06-13,1818.770000,1875.800000,1877.380000,1818.900000,NaN,-0.027900,NaN
2022-08-02,1759.750000,1772.230000,1788.240000,1759.860000,NaN,-0.006800,NaN
2022-12-26,1797.610000,1798.570000,1798.780000,1798.040000,NaN,-0.000200,NaN
2023-01-02,1823.690000,1823.850000,1823.940000,1823.850000,NaN,-0.000400,NaN
2023-06-02,1947.630000,1978.140000,1983.520000,1947.670000,NaN,-0.015300,NaN
2023-06-07,1939.630000,1963.590000,1970.260000,1939.800000,NaN,-0.011800,NaN
2023-12-20,2029.190000,2040.340000,2043.940000,2029.370000,NaN,-0.005400,NaN
2023-12-28,2064.860000,2077.340000,2088.540000,2065.890000,NaN,-0.005900,NaN
2024-07-31,2448.100000,2410.090000,2451.000000,2410.170000,NaN,0.016500,NaN
2025-01-09,2670.650000,2662.200000,2675.420000,2662.300000,NaN,0.003200,NaN


Run All completed. Forecasts generated: True ; diagnostic status: TRAIN_LJUNG_BOX_AND_ROOTS_PASS


### Current execution status

The Step 9 summary above is generated from this XAU execution. It records the chosen model, actual Train diagnostics, Test metrics and future forecasts, or an explicit no-model status. Saved BTC results are not copied into XAU.


### Method references
- [statsmodels ARIMA](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html)
- [ARIMAResults.apply — fixed parameters](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMAResults.apply.html)
- [pmdarima auto_arima](https://alkaline-ml.com/pmdarima/modules/generated/pmdarima.arima.auto_arima.html)
